<img src="https://onstove-documentation.readthedocs.io/en/latest/_images/OnStove_logo_color.svg" height="110" align="center">
&nbsp;&nbsp;
<img src="https://onstove-documentation.readthedocs.io/en/latest/_images/kth_logo.svg" height="65" align="center">
&nbsp;&nbsp;
<img src="https://climatecompatiblegrowth.com/wp-content/uploads/CCG_logo.png" height="65" align="center">

# Welcome to the Starter Data Kit for `OnSTOVE`!

This notebook downloads every geospatial and tabular dataset needed to run an **OnSTOVE** analysis
and packages them into a single **`<ISO3>_SDK.zip`** file ready to use.

### Datasets collected automatically
| Layer | Source | Format |
|---|---|---|
| Administrative boundary | GADM via pygadm | Vector (GPKG) |
| Population 1 km (2020) | WorldPop | Raster (GeoTIFF) |
| Urban-Rural status | GHS-SMOD 2020 (JRC) | Raster (GeoTIFF) |
| Medium-voltage lines | Gridfinder (Zenodo CC-BY) | Vector (GPKG) |
| HV lines (high-voltage power lines only) | OpenStreetMap via Overpass API | Vector (GPKG) |
| Nighttime lights | VIIRS VNL v2.1 2020 (EOG Mines) | Raster (GeoTIFF) |
| Walking friction | MalariaAtlas 2019 | Raster (GeoTIFF) |
| Motorized friction | MalariaAtlas 2019 | Raster (GeoTIFF) |
| Travel time to nearest large town | MalariaAtlas method (own least-cost-path calc on Motorized Friction + GHS-SMOD) | Raster (GeoTIFF) |
| Forest cover | Hansen/GLAD GFC 2020 (GFW tiles) | Raster (GeoTIFF) |
| Forest cover (radar, optional) | JAXA ALOS PALSAR-2 FNF4 2021 via Earth Engine | Raster (GeoTIFF) |
| Livestock density | FAO GLW3 (Zenodo) | Raster (GeoTIFF) |
| Temperature (full GIS package) | Global Solar Atlas v2 GEOTIFF folder | Raster (GeoTIFF, multi-file) |
| Relative Wealth Index | Meta / HDX | CSV |
| Blank socio-economic CSV | Generated from OnSTOVE parameter list | CSV |
| Blank techno-economic CSV | Generated from OnSTOVE parameter list | CSV |
| Specification templates (Technoeconomic + Prep file) | Blank, generic workbooks with Read Me and Parameter Legend tabs - not country data, apply to any country | Excel (XLSX) |

### Layers requiring manual download (no public API)
| Layer | Source |
|---|---|
| Electricity transformers | Country-specific - use [EnergyData.info](https://energydata.info/) or your own national data |

> The Temperature layer is now fetched automatically as the **entire** Global Solar Atlas
> `<Country>_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF` folder (all layers - GHI, DNI,
> DIF, GTI, PVOUT, TEMP, ELE, OPTA - plus metadata), not just the `TEMP.tif` file. If the automatic
> download fails for a given country, fall back to [Global Solar Atlas](https://globalsolaratlas.info/download)
> manually.

### Country coverage
Africa and South/South-East Asia (~60 countries)

---
> **Runtime note:** Gridfinder MV lines (~724 MB) and FAO Livestock are large downloads.
> NTL is fetched from WorldPop Southampton (~30 MB, no auth required).
> Total run time is typically **10–20 minutes** depending on connection speed.


In [1]:
#%%capture
# @title Step 1: Install packages {"vertical-output": true, "display-mode": "form"}
# @markdown Run this cell first. Wait for **Setup complete** before continuing.
# @markdown This may take 2-4 minutes.

import subprocess
from IPython.display import clear_output

def run_quiet(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        clear_output(wait=True)
        print(line.strip())
    p.wait()

run_quiet("pip install --quiet --upgrade pip")
run_quiet("pip install --quiet requests tqdm geopandas pygadm rasterio shapely pyproj pycountry scikit-image")
run_quiet("pip install --quiet ipyleaflet ipywidgets")
# earthengine-api is only needed for the OPTIONAL PALSAR FNF forest-cover layer
# (Step 3). Installed here so it is ready if that box is ticked; authentication
# itself only happens later, and only if that layer is actually requested.
run_quiet("pip install --quiet earthengine-api")

# gdal-bin provides command-line GDAL utilities (e.g. gdalbuildvrt) used
# later for streaming forest cover tiles
run_quiet("apt-get install -y gdal-bin")

import os, requests, zipfile, shutil, subprocess, glob, json, csv
import geopandas as gpd
import pygadm
from tqdm.notebook import tqdm
import ipywidgets as widgets
from ipywidgets import Layout, HBox, VBox, Button
from IPython.display import display, HTML, clear_output
from google.colab import files as cfiles

print("Setup complete")


Processing triggers for man-db (2.10.2-1) ...
Setup complete


In [2]:
#%%capture
# @title Step 2: Load downloader functions {"display-mode": "form"}
# @markdown Run this cell to load all dataset downloader functions.

import os, requests, zipfile, shutil, subprocess, gc
import geopandas as gpd
import pygadm

GEOFABRIK_MAP = {
    'AGO':('africa','angola'),'BDI':('africa','burundi'),'BEN':('africa','benin'),
    'BFA':('africa','burkina-faso'),'BWA':('africa','botswana'),
    'CAF':('africa','central-african-republic'),'CIV':('africa','ivory-coast'),
    'CMR':('africa','cameroon'),'COD':('africa','congo-democratic-republic'),
    'COG':('africa','congo-republic'),'DJI':('africa','djibouti'),
    'DZA':('africa','algeria'),'EGY':('africa','egypt'),'ERI':('africa','eritrea'),
    'ETH':('africa','ethiopia'),'GAB':('africa','gabon'),'GHA':('africa','ghana'),
    'GIN':('africa','guinea'),'GMB':('africa','senegal'),'GNB':('africa','guinea-bissau'),
    'GNQ':('africa','equatorial-guinea'),'KEN':('africa','kenya'),
    'LBR':('africa','liberia'),'LBY':('africa','libya'),'LSO':('africa','lesotho'),
    'MAR':('africa','morocco'),'MDG':('africa','madagascar'),'MLI':('africa','mali'),
    'MOZ':('africa','mozambique'),'MRT':('africa','mauritania'),'MWI':('africa','malawi'),
    'NAM':('africa','namibia'),'NER':('africa','niger'),'NGA':('africa','nigeria'),
    'RWA':('africa','rwanda'),'SDN':('africa','sudan'),'SEN':('africa','senegal'),
    'SLE':('africa','sierra-leone'),'SOM':('africa','somalia'),'SSD':('africa','south-sudan'),
    'SWZ':('africa','swaziland'),'TCD':('africa','chad'),'TGO':('africa','togo'),
    'TUN':('africa','tunisia'),'TZA':('africa','tanzania'),'UGA':('africa','uganda'),
    'ZAF':('africa','south-africa'),'ZMB':('africa','zambia'),'ZWE':('africa','zimbabwe'),
    'BRN':('asia','malaysia-singapore-brunei'),'IDN':('asia','indonesia'),
    'KHM':('asia','cambodia'),'LAO':('asia','laos'),'MMR':('asia','myanmar'),
    'MYS':('asia','malaysia-singapore-brunei'),'PHL':('asia','philippines'),
    'SGP':('asia','malaysia-singapore-brunei'),'THA':('asia','thailand'),
    'TLS':('asia','east-timor'),'VNM':('asia','vietnam'),
}

GRIDFINDER_URL   = 'https://zenodo.org/records/3628142/files/grid.gpkg?download=1'
GRIDFINDER_LOCAL = '/tmp/gridfinder_global.gpkg'

def _stream(url, dest, label='', timeout=600):
    h = {'User-Agent': 'OnSTOVE-SDK/1.0'}
    r = requests.get(url, stream=True, timeout=timeout, headers=h)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    done = 0
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8*1024*1024):
            f.write(chunk); done += len(chunk)
            if total:
                print(f'\r  {label} {done/total*100:.1f}%  ({done//1048576}/{total//1048576} MB)',
                      end='', flush=True)
    print()

# Session-level cache: {country: GeoDataFrame}. Populated either by a
# successful GADM fetch or by a manually uploaded boundary file (see
# set_manual_boundary / the upload widget in Step 3). Every downloader that
# needs the country outline (clipping, bounding boxes, etc.) calls _boundary(),
# so without this cache each one would re-hit GADM independently - this is
# why errors could persist even after the retry/backoff was added.
_BOUNDARY_CACHE = {}

def set_manual_boundary(country, gdf):
    """Register a manually supplied boundary GeoDataFrame for a country so
    _boundary()/get_boundaries() use it instead of fetching from GADM."""
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    else:
        gdf = gdf.to_crs(4326)
    _BOUNDARY_CACHE[country] = gdf
    return gdf

def _boundary(country, _retries=3, _base_wait=5):
    """Return the country boundary, using the session cache (auto-fetched or
    manually uploaded) if available, otherwise fetching from GADM with
    retry/backoff since the GADM server (geodata.ucdavis.edu) frequently
    stalls or times out under load, especially from cloud IP ranges like
    Colab's. This is the single most-called helper in the notebook (used by
    nearly every downloader via clipping/bounding-box), so caching + retrying
    here fixes GADM flakiness everywhere at once."""
    if country in _BOUNDARY_CACHE:
        return _BOUNDARY_CACHE[country]
    import time
    import requests as _requests
    last_err = None
    for attempt in range(1, _retries + 1):
        try:
            b = pygadm.Items(admin=country, content_level=0)
            b = b.set_crs(4326, allow_override=True)
            _BOUNDARY_CACHE[country] = b
            return b
        except (_requests.exceptions.ConnectTimeout,
                _requests.exceptions.ConnectionError,
                _requests.exceptions.ReadTimeout,
                _requests.exceptions.Timeout) as e:
            last_err = e
            if attempt < _retries:
                wait = _base_wait * attempt  # increasing backoff: 5s, 10s, 15s ...
                print(f'[Boundaries] GADM server timed out (attempt {attempt}/{_retries}). '
                      f'Retrying in {wait}s ...')
                time.sleep(wait)
            else:
                print(f'[Boundaries] GADM server still unreachable after {_retries} attempts.')
        except Exception as e:
            # Non-network errors (bad country code etc.) - fail fast, no point retrying.
            raise
    raise Exception(
        f'Could not reach the GADM server for {country} after {_retries} attempts '
        f'(last error: {last_err}). This is usually a transient GADM outage/rate-limit - '
        f'try again in a few minutes, Runtime > Restart runtime and re-run, or use the '
        f'"Upload boundary manually" option in Step 3 to supply the file yourself.'
    )

def _mkdir(p):
    os.makedirs(p, exist_ok=True); return p

def _clip_rasterio(src_path, country, dest_path):
    import rasterio
    from rasterio.mask import mask as rio_mask
    b = _boundary(country)
    with rasterio.open(src_path) as src:
        b2 = b.to_crs(src.crs)
        out_img, out_tf = rio_mask(src, b2.geometry, crop=True)
        meta = src.meta.copy()
        meta.update({'driver':'GTiff','height':out_img.shape[1],
                     'width':out_img.shape[2],'transform':out_tf})
        with rasterio.open(dest_path, 'w', **meta) as dst:
            dst.write(out_img)

def _clip(src_path, country, dest_path, nodata=-9999):
    b   = _boundary(country)
    tmp = f'/tmp/{country}_bnd.gpkg'
    b.to_file(tmp, driver='GPKG')
    cmd = (f'gdalwarp -cutline {tmp} -crop_to_cutline -dstnodata {nodata} '
           f'-t_srs EPSG:4326 -of GTiff {src_path} {dest_path}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(dest_path):
        _clip_rasterio(src_path, country, dest_path)

# --- 1. Boundary ---
def get_boundaries(country):
    out = _mkdir(f'Data/{country}/Boundaries')
    p   = f'{out}/{country}_boundary.gpkg'
    if country in _BOUNDARY_CACHE:
        print(f'[Boundaries] Using manually uploaded boundary for {country} ...')
    else:
        print(f'[Boundaries] Fetching GADM boundary for {country} ...')
    _boundary(country).to_file(p, driver='GPKG')
    print(f'[Boundaries] Saved -> {p}')

# --- 2. Population ---
def get_population(country):
    out  = _mkdir(f'Data/{country}/Population')
    dest = f'{out}/{country}_population_2020.tif'
    url  = (f'https://data.worldpop.org/GIS/Population/Global_2000_2020_1km/'
            f'2020/{country}/{country.lower()}_ppp_2020_1km_Aggregated.tif')
    print(f'[Population] Downloading WorldPop 2020 for {country} ...')
    try:
        _stream(url, dest, label='[Pop]')
    except Exception:
        url2 = (f'https://data.worldpop.org/GIS/Population/Global_2000_2020/'
                f'2020/{country}/{country.lower()}_ppp_2020.tif')
        print('[Population] Trying 100m fallback ...')
        _stream(url2, dest, label='[Pop fallback]')
    print(f'[Population] Saved -> {dest}')

# --- 3. GHS-SMOD Urban-Rural ---
def get_urban_rural(country):
    out    = _mkdir(f'Data/{country}/UrbanRural')
    dest   = f'{out}/{country}_ghs_smod_2020.tif'
    global_tif = '/tmp/ghsl_smod_global.tif'
    if not os.path.exists(global_tif):
        url = ('https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/'
               'GHS_SMOD_GLOBE_R2023A/GHS_SMOD_E2020_GLOBE_R2023A_54009_1000/'
               'V1-0/GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V1_0.zip')
        tmp_zip = '/tmp/ghsl_smod.zip'
        print('[Urban-Rural] Downloading GHS-SMOD global (~160 MB) ...')
        _stream(url, tmp_zip, label='[SMOD]')
        with zipfile.ZipFile(tmp_zip, 'r') as z:
            tifs = [n for n in z.namelist() if n.endswith('.tif')]
            z.extract(tifs[0], '/tmp/ghsl_extract/')
            shutil.move(f'/tmp/ghsl_extract/{tifs[0]}', global_tif)
    _clip(global_tif, country, dest, nodata=0)
    print(f'[Urban-Rural] Saved -> {dest}')

# --- 4. MV lines (Gridfinder) ---
def _is_valid_gpkg(path):
    """A GeoPackage is a SQLite database - check the SQLite magic header AND
    that SQLite can actually open it, rather than trusting file size alone.
    A partially-written or corrupted cached download will pass a size check
    but fail here, which is exactly what caused 'database disk image is
    malformed' errors downstream when GeoPandas/Fiona tried to read it."""
    import sqlite3
    try:
        with open(path, 'rb') as f:
            if f.read(16) != b'SQLite format 3\x00':
                return False
        con = sqlite3.connect(f'file:{path}?mode=ro', uri=True)
        con.execute('PRAGMA quick_check;').fetchone()
        con.close()
        return True
    except Exception:
        return False

def get_mv_lines(country):
    out  = _mkdir(f'Data/{country}/MV_lines')
    dest = f'{out}/{country}_mv_lines.gpkg'
    need_download = not (os.path.exists(GRIDFINDER_LOCAL) and
                          os.path.getsize(GRIDFINDER_LOCAL) > 100*1024*1024)
    if not need_download and not _is_valid_gpkg(GRIDFINDER_LOCAL):
        print('[MV lines] Cached Gridfinder file is corrupted/incomplete - re-downloading ...')
        os.remove(GRIDFINDER_LOCAL)
        need_download = True
    if need_download:
        print('[MV lines] Downloading Gridfinder global (~724 MB) - once per session ...')
        _stream(GRIDFINDER_URL, GRIDFINDER_LOCAL, label='[Gridfinder]')
        if not _is_valid_gpkg(GRIDFINDER_LOCAL):
            os.remove(GRIDFINDER_LOCAL)
            raise Exception('Downloaded Gridfinder file failed validation (corrupted or incomplete '
                             'download) and was removed. Please retry.')
    b    = _boundary(country)
    bbox = tuple(b.total_bounds)
    print(f'[MV lines] Clipping to {country} ...')
    try:
        gdf = gpd.read_file(GRIDFINDER_LOCAL, bbox=bbox).set_crs(4326, allow_override=True)
    except Exception as e:
        # Belt-and-braces: if the cached file somehow still fails to read
        # (e.g. corrupted after the check above, mid-session disk issue),
        # remove it so the NEXT retry re-downloads cleanly instead of
        # repeatedly hitting the same broken file.
        if os.path.exists(GRIDFINDER_LOCAL):
            os.remove(GRIDFINDER_LOCAL)
        raise Exception(f'Gridfinder file could not be read ({e}) - cached copy removed, please retry.')
    gdf  = gpd.clip(gdf, b.to_crs(gdf.crs))
    gdf.to_file(dest, driver='GPKG')
    print(f'[MV lines] Saved {len(gdf)} features -> {dest}')

# --- 5. HV lines - HIGH-VOLTAGE POWER LINES ONLY, via OSM Overpass ---
# NOTE ON THIS LAYER - READ BEFORE CHANGING:
# This function fetches ONLY high-voltage (HV) electricity TRANSMISSION
# lines (OSM tag power=line). It does NOT fetch roads, railways, medium/
# minor voltage lines, substations, towers, or any other infrastructure -
# HV lines is the ONLY thing this layer represents, matching its label.
#
# This is a distance-to-grid proxy used by OnSTOVE/OnSSET, and is unrelated
# to the road network. Geofabrik's free .shp.zip extracts only cover a fixed
# set of themes (roads, buildings, landuse, railways, waterways, natural,
# places, pois, transport) - power infrastructure is NOT one of them
# (Geofabrik only sells a custom "power export" on request). An earlier
# version of this function filtered the ROADS shapefile by
# motorway/trunk/primary class and saved the result as "HV lines" - that was
# not a memory problem but a genuine labelling/data bug, silently saving a
# subset of the ROAD network under this layer's name. That has been fixed:
# the function below queries OSM's actual power=line tag via the Overpass
# API, and only power=line (major/high-voltage transmission) - power=minor_line
# (local/lower-voltage distribution) is deliberately excluded, so this stays
# strictly a HIGH-voltage layer. Querying the power tag directly also means
# only the power features themselves are ever downloaded, not a whole
# national roads layer, which fixes the RAM issue for this layer too.
# Where available, https://energydata.info hosts better-curated, country-
# specific transmission network shapefiles (voltage-graded, validated against
# utility data) - there's no consistent URL pattern to fetch these
# automatically per-country, so that's listed as a manual-fallback source in
# the Instructions file, the same way Electricity Transformers already is.
_OVERPASS_ENDPOINTS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.private.coffee/api/interpreter',
    'https://overpass.nchc.org.tw/api/interpreter',
    'https://overpass.osm.ch/api/interpreter',
]

def get_hv_lines(country):
    import shapely.geometry as sgeom, time
    out  = _mkdir(f'Data/{country}/HV_lines')
    dest = f'{out}/{country}_hv_lines.gpkg'
    b    = _boundary(country)
    bb   = b.total_bounds  # (minx, miny, maxx, maxy) = (west, south, east, north)
    bbox_ov = f'{bb[1]:.5f},{bb[0]:.5f},{bb[3]:.5f},{bb[2]:.5f}'  # Overpass wants south,west,north,east

    # power=line is OSM's tag for major (high-voltage transmission) lines;
    # power=minor_line is lower-voltage local distribution and is deliberately
    # excluded here to keep this layer to HIGH-voltage lines only.
    query = (
        f'[out:json][timeout:180][bbox:{bbox_ov}];'
        '(way["power"="line"];);'
        'out geom;'
    )
    # Overpass mirrors sometimes rate-limit anonymous/default requests library
    # User-Agents more aggressively - identify ourselves properly.
    headers = {'User-Agent': 'OnSTOVE-SDK/1.0 (research use; contact: your-email@example.com)'}

    data, last_err = None, None
    for ep in _OVERPASS_ENDPOINTS:
        # One retry per endpoint on 429, honouring the server's Retry-After
        # header, before moving on to the next mirror. Colab runtimes share
        # IP ranges with many other users, so 429s are frequently about the
        # shared IP being throttled rather than this query specifically.
        for attempt in range(2):
            try:
                print(f"[HV lines] Querying OSM power lines via Overpass ({ep.split('/')[2]}) attempt {attempt + 1} ...")
                r = requests.post(ep, data={'data': query}, headers=headers, timeout=180)
                if r.status_code == 429:
                    wait = int(r.headers.get('Retry-After', 30))
                    print(f'[HV lines] 429 (rate limited) from {ep}, waiting {wait}s before retry ...')
                    time.sleep(wait)
                    last_err = f'429 Too Many Requests (waited {wait}s, still failed)'
                    continue
                r.raise_for_status()
                data = r.json()
                break
            except Exception as e:
                last_err = e
                print(f'[HV lines] {ep} failed ({e}), trying next endpoint ...')
        if data is not None:
            break
    if data is None:
        raise Exception(f'All Overpass endpoints failed - last error: {last_err}')

    rows = []
    for el in data.get('elements', []):
        if el.get('type') != 'way' or 'geometry' not in el or len(el['geometry']) < 2:
            continue
        coords = [(pt['lon'], pt['lat']) for pt in el['geometry']]
        tags = el.get('tags', {})
        rows.append({
            'osm_id':  el.get('id'),
            'power':   tags.get('power'),
            'voltage': tags.get('voltage'),
            'geometry': sgeom.LineString(coords),
        })
    if not rows:
        # No high-voltage lines mapped in OSM for this country - write an
        # empty (but validly-structured) GeoPackage rather than skipping, so
        # downstream OnSTOVE steps that expect this file to exist don't break.
        print(f'[HV lines] No OSM high-voltage power lines found for {country} - saving empty file.')
        gpd.GeoDataFrame(columns=['osm_id', 'power', 'voltage', 'geometry'],
                          geometry='geometry', crs=4326).to_file(dest, driver='GPKG')
        print(f'[HV lines] Saved (empty) -> {dest}')
        return

    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs=4326)
    gpd.clip(gdf, b).to_file(dest, driver='GPKG')
    print(f'[HV lines] Saved {len(gdf)} features -> {dest}')
    del gdf
    gc.collect()

# --- 6. Nighttime Lights (WorldPop Global2 series — data.worldpop.org) ---
# URL confirmed from WorldPop manifest (worldpoppy package, product: ntl_viirs_g2).
# Pattern: https://data.worldpop.org/GIS/Covariates/Global_2015_2030/
#           {ISO3}/VIIRS/v1/nvf/{iso3}_viirs_nvf_{year}_100m_v1.tif
# - Country-clipped TIF, ~30 MB, 100m resolution, no auth required
# - NVF = No Vegetation Filtering (correct for electrification proxy)
# - Available 2015–2023
_NTL_YEAR = 2020   # change to any year 2015–2023 if needed
_NTL_BASE = 'https://data.worldpop.org/GIS/Covariates/Global_2015_2030'

def get_nighttime_lights(country):
    out   = _mkdir(f'Data/{country}/NighttimeLights')
    dest  = f'{out}/{country}_ntl_viirs_{_NTL_YEAR}.tif'
    iso3  = country.upper()
    iso3l = country.lower()
    url   = (f'{_NTL_BASE}/{iso3}/VIIRS/v1/nvf/'
             f'{iso3l}_viirs_nvf_{_NTL_YEAR}_100m_v1.tif')
    print(f'[NTL] Downloading WorldPop VIIRS NTL {_NTL_YEAR} for {iso3} (~30 MB) ...')
    _stream(url, dest, label='[NTL]')
    print(f'[NTL] Saved -> {dest}')

# --- 7 & 8. Walking + Motorized friction (data.malariaatlas.org WCS) ---
# The old malariaatlas.org/geoserver/ows endpoint was decommissioned (404).
# MalariaAtlas migrated to a new server: data.malariaatlas.org/geoserver
# with updated January 2020 friction surfaces (coverageId prefix: 202001).
# These are fetched as bbox-clipped GeoTIFFs — no global download needed.
_MAP_WCS_BASE = 'https://data.malariaatlas.org/geoserver/Accessibility/wcs'

def _friction_wcs_url(coverage_id, bb):
    """Build a WCS 2.0.1 GetCoverage URL clipped to a country bounding box."""
    return (
        f'{_MAP_WCS_BASE}?service=WCS&version=2.0.1&request=GetCoverage'
        f'&coverageId={coverage_id}'
        f'&format=image/tiff'
        f'&subset=Long({bb[0]:.4f},{bb[2]:.4f})'
        f'&subset=Lat({bb[1]:.4f},{bb[3]:.4f})'
    )

def get_walking_friction(country):
    out  = _mkdir(f'Data/{country}/WalkingFriction')
    dest = f'{out}/{country}_walking_friction.tif'
    b    = _boundary(country); bb = b.total_bounds
    url  = _friction_wcs_url('202001_Global_Walking_Only_Friction_Surface', bb)
    print('[Walking Friction] Downloading from data.malariaatlas.org ...')
    _stream(url, dest, label='[Walk]')
    print(f'[Walking Friction] Saved -> {dest}')

# --- 8. Motorized friction (data.malariaatlas.org WCS) ---
def get_motorized_friction(country):
    out  = _mkdir(f'Data/{country}/MotorizedFriction')
    dest = f'{out}/{country}_motorized_friction.tif'
    b    = _boundary(country); bb = b.total_bounds
    url  = _friction_wcs_url('202001_Global_Motorized_Friction_Surface', bb)
    print('[Motorized Friction] Downloading from data.malariaatlas.org ...')
    _stream(url, dest, label='[Moto]')
    print(f'[Motorized Friction] Saved -> {dest}')

# --- 8b. Travel time to nearest large town (MalariaAtlas method) ---
# Replicates the method behind the Malaria Atlas Project's "Accessibility to
# Cities" surface (Weiss et al. 2018, Nature, doi:10.1038/nature25181):
#   1. "Large towns" are MAP's own definition - contiguous built-up areas with
#      >=1500 people/km2 (or majority built-up land cover) and a total cluster
#      population >=50,000. This is exactly the JRC GHS-SMOD "Urban Centre"
#      (class 30) definition, so the Urban-Rural layer already fetched above
#      supplies the town locations - no extra dataset needed.
#   2. The Motorized Friction surface (minutes to travel 1 metre) is converted
#      into a per-pixel cost (minutes to cross that cell) using each cell's
#      real-world size in metres.
#   3. A multi-source Dijkstra least-cost accumulation - the same family of
#      cost-distance algorithm MAP used on their friction surface - is run
#      from every Urban Centre cell at once, giving the cumulative travel
#      time from every pixel to its NEAREST large town.
# In OnSTOVE this is used as a proxy for travel time to LPG supply points,
# since LPG resupply infrastructure tends to sit in or near larger towns.
def get_travel_time_to_towns(country):
    import numpy as np
    import rasterio
    from rasterio.warp import reproject, Resampling
    from pyproj import Geod

    out  = _mkdir(f'Data/{country}/TravelTimeToTowns')
    dest = f'{out}/{country}_travel_time_to_towns.tif'

    friction_path = f'Data/{country}/MotorizedFriction/{country}_motorized_friction.tif'
    smod_path     = f'Data/{country}/UrbanRural/{country}_ghs_smod_2020.tif'
    if not os.path.exists(friction_path):
        print('[Travel Time] Motorized friction not found - fetching it first ...')
        get_motorized_friction(country)
    if not os.path.exists(smod_path):
        print('[Travel Time] Urban-Rural (GHS-SMOD) not found - fetching it first ...')
        get_urban_rural(country)

    print('[Travel Time] Computing least-cost travel time to nearest large town (MalariaAtlas method) ...')

    with rasterio.open(friction_path) as fsrc:
        # float32 instead of float64 - halves the memory of every full-country
        # array built below (friction, cost, valid, smod_on_friction, travel_time)
        # with no meaningful loss of precision for minutes-per-cell values.
        friction    = fsrc.read(1).astype('float32')
        f_transform = fsrc.transform
        f_crs       = fsrc.crs
        f_shape     = fsrc.shape
        f_nodata    = fsrc.nodata

        # Resample GHS-SMOD onto the friction grid (nearest-neighbour - it's categorical)
        with rasterio.open(smod_path) as ssrc:
            smod_on_friction = np.zeros(f_shape, dtype='float32')
            reproject(
                source=rasterio.band(ssrc, 1),
                destination=smod_on_friction,
                src_transform=ssrc.transform, src_crs=ssrc.crs,
                dst_transform=f_transform, dst_crs=f_crs,
                resampling=Resampling.nearest
            )

    # GHS-SMOD class 30 = 'Urban Centre' -> MAP's large-town definition (>=50k, >=1500/km2)
    town_mask = (smod_on_friction == 30)
    if not town_mask.any():
        print('[Travel Time] No GHS Urban Centre (50k+) found - falling back to Dense Urban Cluster.')
        town_mask = (smod_on_friction == 23)  # class 23 = Dense Urban Cluster
    if not town_mask.any():
        print(f'[Travel Time] No urban settlements found for {country} - skipping.')
        return

    # Convert friction (minutes / metre) into a per-pixel cost (minutes to cross the
    # cell) using each row's true cell size in metres (accounts for latitude).
    geod = Geod(ellps='WGS84')
    height, width = f_shape
    cell_size_m = np.zeros(height)
    for row in range(height):
        lon0, lat0 = f_transform * (0, row)
        lon1, _    = f_transform * (1, row)
        lon2, lat2 = f_transform * (0, row + 1)
        _, _, dx = geod.inv(lon0, lat0, lon1, lat0)
        _, _, dy = geod.inv(lon0, lat0, lon2, lat2)
        cell_size_m[row] = (abs(dx) + abs(dy)) / 2
    cell_size_m = np.repeat(cell_size_m[:, None], width, axis=1)

    cost  = friction * cell_size_m
    valid = np.isfinite(friction) & (friction > 0)
    if f_nodata is not None:
        valid &= (friction != f_nodata)
    cost[~valid]  = np.inf
    cost[town_mask] = 0  # towns are zero-cost sources

    # Multi-source Dijkstra least-cost accumulation (cost-distance), the same
    # class of algorithm MAP used to build the Accessibility to Cities surface.
    try:
        from skimage.graph import MCP_Geometric
    except ImportError:
        run_quiet('pip install --quiet scikit-image')
        from skimage.graph import MCP_Geometric

    mcp = MCP_Geometric(cost, fully_connected=True)
    starts = list(zip(*np.where(town_mask)))
    travel_time, _ = mcp.find_costs(starts)
    travel_time[~valid] = np.nan

    out_arr = np.where(np.isfinite(travel_time), travel_time, -9999).astype('float32')
    meta = dict(driver='GTiff', height=height, width=width, count=1,
                dtype='float32', crs=f_crs, transform=f_transform, nodata=-9999)
    with rasterio.open(dest, 'w', **meta) as dst:
        dst.write(out_arr, 1)
    n_towns = int(town_mask.sum())
    del friction, smod_on_friction, cost, valid, cell_size_m, town_mask, travel_time, out_arr, mcp
    gc.collect()
    print(f'[Travel Time] Saved -> {dest} (minutes to nearest large town, {n_towns} town cells used)')

# --- 9. Forest cover (Hansen GFC 2020 tiles) ---
# Tile naming confirmed from Hansen download.html:
# Format is {num}{N/S}_{num}{E/W}, e.g. 10N_000E (NOT N10_E000).
# Tiles are 10-degree squares named by their TOP-LEFT corner.
def get_forest_cover(country):
    out  = _mkdir(f'Data/{country}/ForestCover')
    dest = f'{out}/{country}_forest_cover_2020.tif'
    b    = _boundary(country); bb = b.total_bounds
    tile_lats = range(int((bb[1]//10)*10), int((bb[3]//10)*10)+10, 10)
    tile_lons = range(int((bb[0]//10)*10), int((bb[2]//10)*10)+10, 10)
    tile_files = []
    for tlat in tile_lats:
        for tlon in tile_lons:
            lat_top = tlat + 10
            lat_s = str(abs(lat_top)).zfill(2) + ('N' if lat_top >= 0 else 'S')
            lon_s = str(abs(tlon)).zfill(3)    + ('E' if tlon   >= 0 else 'W')
            tname = f'Hansen_GFC-2020-v1.8_treecover2000_{lat_s}_{lon_s}.tif'
            url   = f'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2020-v1.8/{tname}'
            tmp   = f'/tmp/{tname}'
            if not os.path.exists(tmp):
                try:
                    print(f'[Forest] Downloading tile {lat_s}_{lon_s} ...')
                    _stream(url, tmp, label='[Forest]')
                    tile_files.append(tmp)
                except Exception as e:
                    print(f'[Forest] Tile {lat_s}_{lon_s} not available: {e}')
            else:
                print(f'[Forest] Using cached tile {lat_s}_{lon_s}.')
                tile_files.append(tmp)
    if not tile_files:
        print('[Forest] No tiles found for this country.'); return
    if len(tile_files) == 1:
        _clip(tile_files[0], country, dest)
    else:
        # Each Hansen tile is a 10-degree, 30 m-resolution GeoTIFF (~1.5-1.6 GB
        # as a raw array). The old code opened every tile with rasterio.merge,
        # which reads them ALL fully into memory and concatenates them into one
        # giant numpy mosaic before clipping - for any country spanning two or
        # more tiles (most of them) that alone was enough to exhaust Colab's
        # RAM. A GDAL VRT is just a small XML index of the tiles on disk;
        # gdalbuildvrt never reads pixel data, and gdalwarp then streams the
        # clip block-by-block straight off disk, so the full mosaic is never
        # held in memory at once.
        vrt_path = '/tmp/forest_mosaic.vrt'
        r = subprocess.run(['gdalbuildvrt', '-q', vrt_path] + tile_files,
                            capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists(vrt_path):
            raise Exception(f'gdalbuildvrt failed: {r.stderr}')
        _clip(vrt_path, country, dest)
        if os.path.exists(vrt_path): os.remove(vrt_path)
    print(f'[Forest] Saved -> {dest}')
    gc.collect()

# --- 9b. Forest cover (JAXA ALOS PALSAR-2 FNF4, radar - OPTIONAL, via Earth Engine) ---
# Alternative to the Hansen/GLAD layer above. PALSAR is L-band SAR (radar),
# not optical, so it sees through cloud/haze - useful for persistently cloudy
# tropical regions (e.g. the Congo Basin) where Landsat-derived products like
# Hansen can have data gaps. Output here is categorical (Dense Forest /
# Non-dense Forest / Non-Forest / Water), 25 m resolution, most recent
# available year (2021) of the JAXA/ALOS/PALSAR/YEARLY/FNF4 collection.
# Requires a one-time Earth Engine login (opens a browser tab/URL) and a
# Google Cloud project registered for Earth Engine access - this is the only
# layer in the kit that needs authentication, which is why it's optional.
# JAXA prohibits commercial use of PALSAR/FNF data without their consent -
# fine for research/planning use, but flag this if the output goes anywhere
# commercial. See https://developers.google.com/earth-engine/datasets/catalog/JAXA_ALOS_PALSAR_YEARLY_FNF4
_EE_INITIALIZED = False

def _ee_init(project_id=None):
    global _EE_INITIALIZED
    if _EE_INITIALIZED:
        return
    import ee
    try:
        ee.Initialize(project=project_id) if project_id else ee.Initialize()
    except Exception:
        print('[Earth Engine] Not authenticated yet - opening the login flow '
              '(follow the prompt/link, then paste the code back here) ...')
        ee.Authenticate()
        ee.Initialize(project=project_id) if project_id else ee.Initialize()
    _EE_INITIALIZED = True

def get_forest_cover_palsar_fnf(country, ee_project=None):
    import ee
    _ee_init(ee_project)

    out  = _mkdir(f'Data/{country}/ForestCover')
    dest = f'{out}/{country}_forest_cover_palsar_fnf_2021.tif'
    b    = _boundary(country); bb = b.total_bounds  # (west, south, east, north)

    fnf = (ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/FNF4')
             .filterDate('2021-01-01', '2021-12-31')
             .select('fnf').mosaic())

    # Request in a grid of ~1-degree tiles rather than one call for the whole
    # country - Earth Engine's synchronous getDownloadURL caps out well below
    # what a large country needs at 25 m resolution, and tiling (same pattern
    # as the Hansen fix above) keeps every individual download small so this
    # never has to hold a country-sized array in memory either.
    import math
    lon0, lat0 = math.floor(bb[0]), math.floor(bb[1])
    lon1, lat1 = math.ceil(bb[2]), math.ceil(bb[3])

    tile_files = []
    for tlon in range(lon0, lon1):
        for tlat in range(lat0, lat1):
            region = ee.Geometry.Rectangle([tlon, tlat, tlon + 1, tlat + 1])
            tmp = f'/tmp/{country}_fnf_{tlon}_{tlat}.tif'
            try:
                url = fnf.getDownloadURL({
                    'region': region, 'scale': 25, 'crs': 'EPSG:4326',
                    'format': 'GEO_TIFF',
                })
                r = requests.get(url, timeout=120)
                r.raise_for_status()
                if len(r.content) > 0:
                    with open(tmp, 'wb') as f:
                        f.write(r.content)
                    tile_files.append(tmp)
            except Exception as e:
                # Tiles with no data (open ocean, outside the FNF mask, etc.)
                # routinely fail/come back empty - that's expected, not an error.
                print(f'[Forest PALSAR] Tile {tlon},{tlat} skipped ({e})')

    if not tile_files:
        print(f'[Forest PALSAR] No tiles retrieved for {country} - skipping.')
        return

    if len(tile_files) == 1:
        _clip(tile_files[0], country, dest, nodata=0)
    else:
        vrt_path = f'/tmp/{country}_fnf_mosaic.vrt'
        r = subprocess.run(['gdalbuildvrt', '-q', vrt_path] + tile_files,
                            capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists(vrt_path):
            raise Exception(f'gdalbuildvrt failed: {r.stderr}')
        _clip(vrt_path, country, dest, nodata=0)
        if os.path.exists(vrt_path): os.remove(vrt_path)

    for t in tile_files:
        if os.path.exists(t):
            try: os.remove(t)
            except OSError: pass
    print(f'[Forest PALSAR] Saved -> {dest} (1=Dense Forest, 2=Non-dense Forest, 3=Non-Forest, 4=Water)')
    gc.collect()

# --- 10. Livestock (FAO GLW4 2020 — GCS direct URLs) ---
# GLW4 2020 global TIFs from FAO GIS manager Google Cloud Storage bucket.
# FAO catalog URLs (data.apps.fao.org) redirect through a CKAN auth layer
# returning HTML instead of TIF. GCS URLs are direct, no auth required.
# Species codes: BFL confirmed (AmeriGEOSS); CTL/GTS/SHP confirmed on GCS.
_GLW4_GCS = 'https://storage.googleapis.com/fao-gismgr-glw4-2020-data/DATA/GLW4-2020/MAPSET/D-DA'

_GLW4_LAYERS = {
    'cattle': f'{_GLW4_GCS}/GLW4-2020.D-DA.CTL.tif',
    'goats':  f'{_GLW4_GCS}/GLW4-2020.D-DA.GTS.tif',
    'sheep':  f'{_GLW4_GCS}/GLW4-2020.D-DA.SHP.tif',
}

def get_livestock(country):
    out = _mkdir(f'Data/{country}/Livestock')
    for animal, url in _GLW4_LAYERS.items():
        dest    = f'{out}/{country}_livestock_{animal}.tif'
        tmp_tif = f'/tmp/glw4_{animal}_global.tif'
        if not (os.path.exists(tmp_tif) and os.path.getsize(tmp_tif) > 5_000_000):
            print(f'[Livestock] Downloading GLW4 2020 {animal} from FAO GCS ...')
            r = requests.get(url, stream=True, headers={'User-Agent': 'OnSTOVE-SDK/1.0'}, timeout=60)
            r.raise_for_status()
            ct = r.headers.get('Content-Type', '')
            if 'html' in ct.lower():
                raise ValueError(f'[Livestock] Got HTML for {animal} — species code may need updating.')
            with open(tmp_tif, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk: f.write(chunk)
        else:
            print(f'[Livestock] Using cached GLW4 {animal}.')
        print(f'[Livestock] Clipping {animal} to {country} ...')
        _clip(tmp_tif, country, dest)
        print(f'[Livestock] Saved {animal} -> {dest}')

# --- 11. Wealth Index (Meta RWI via HDX CKAN API + WorldPop fallback) ---
# HDX returns HTTP 202 for the direct per-country download URL — this is their
# consent/interest survey gate which blocks programmatic access.
# Fix: use the HDX CKAN resource_show API to fetch the actual S3 storage URL
# for the full 93-country zip, bypassing the gate entirely.
# HDX dataset UUID: 76f2a2ea-ba50-40f5-b79c-db95d668b843
# Full zip resource UUID: de2f953e-940c-43bb-b1f8-4d02d28124b5
_HDX_PACKAGE_API = 'https://data.humdata.org/api/3/action/package_show'
_HDX_DATASET_ID  = '76f2a2ea-ba50-40f5-b79c-db95d668b843'  # relative-wealth-index dataset
_RWI_ZIP_UUID    = 'de2f953e-940c-43bb-b1f8-4d02d28124b5'
_RWI_ZIP_TMP     = '/tmp/rwi_93countries.zip'
_RWI_ZIP_EXT     = '/tmp/rwi_93countries/'

def _is_zip(path):
    """Check the zip magic bytes (PK\x03\x04 or PK\x05\x06) rather than trusting the extension."""
    try:
        with open(path, 'rb') as f:
            sig = f.read(4)
        return sig[:2] == b'PK'
    except Exception:
        return False

def get_wealth_index(country):
    import pycountry, zipfile as zf
    out  = _mkdir(f'Data/{country}/WealthIndex')
    try:
        c_info = pycountry.countries.get(alpha_3=country)
        iso2 = c_info.alpha_2.lower()
        cname = c_info.name.split(',')[0]  # e.g. "Burundi", "Congo" (drop ", Dem. Rep." style suffixes)
    except Exception:
        iso2, cname = country[:2].lower(), country

    dest_rwi = f'{out}/{country}_relative_wealth_index.csv'

    # Attempt 1: look for this country's OWN resource on the HDX dataset (many countries
    # were added individually after the original 93-country package, as e.g.
    # "Burundi_relative_wealth_index.csv" - this is smaller, faster, and sidesteps any
    # issue with the big zip entirely).
    print(f'[Wealth Index] Checking HDX for a per-country RWI resource for {cname} ...')
    try:
        pkg_r = requests.get(
            _HDX_PACKAGE_API,
            params={'id': _HDX_DATASET_ID},
            headers={'User-Agent': 'OnSTOVE-SDK/1.0'},
            timeout=30
        )
        pkg_r.raise_for_status()
        resources = pkg_r.json()['result']['resources']
        match = next(
            (r for r in resources
             if r.get('name', '').lower().startswith(cname.lower() + '_relative_wealth_index')),
            None
        )
        if match:
            print(f'[Wealth Index] Found per-country resource: {match["name"]}')
            _stream(match['url'], dest_rwi, label='[RWI]')
            print(f'[Wealth Index] Meta RWI saved -> {dest_rwi}')
            return
        else:
            print(f'[Wealth Index] No per-country resource named for {cname}; trying the 93-country package ...')
    except Exception as e:
        print(f'[Wealth Index] Per-country lookup failed: {e}')

    # Attempt 2: Meta RWI 93-country zip via HDX CKAN API (bypasses the 202 consent gate)
    try:
        res_r = requests.get(
            'https://data.humdata.org/api/3/action/resource_show',
            params={'id': _RWI_ZIP_UUID},
            headers={'User-Agent': 'OnSTOVE-SDK/1.0'},
            timeout=30
        )
        res_r.raise_for_status()
        s3_url = res_r.json()['result']['url']
        print(f'[Wealth Index] Got direct URL: {s3_url[:80]}...')
        if not (os.path.exists(_RWI_ZIP_TMP) and _is_zip(_RWI_ZIP_TMP)):
            print('[Wealth Index] Downloading 93-country RWI zip (~35 MB) ...')
            _stream(s3_url, _RWI_ZIP_TMP, label='[RWI]')
        if not _is_zip(_RWI_ZIP_TMP):
            with open(_RWI_ZIP_TMP, 'rb') as f:
                head = f.read(200)
            raise Exception(f'Downloaded file is not a zip - server likely returned an error/HTML page '
                             f'instead of the file (first bytes: {head[:120]!r})')
        if not os.path.exists(_RWI_ZIP_EXT):
            with zf.ZipFile(_RWI_ZIP_TMP, 'r') as z: z.extractall(_RWI_ZIP_EXT)
        # Find the CSV for this country's iso2
        matches = [f for f in os.listdir(_RWI_ZIP_EXT)
                   if f.lower().endswith('.csv') and iso2 in f.lower()]
        if matches:
            import shutil
            shutil.copy(f'{_RWI_ZIP_EXT}{matches[0]}', dest_rwi)
            print(f'[Wealth Index] Meta RWI saved -> {dest_rwi}')
            return
        else:
            print(f'[Wealth Index] {country} ({iso2}) not in RWI 93-country package.')
    except Exception as e:
        print(f'[Wealth Index] HDX CKAN approach failed: {e}')

    # Attempt 3: WorldPop poverty index (TIF, ~100m) - try a few common years since
    # coverage/year varies by country.
    dest_wp = f'{out}/{country}_worldpop_poverty_index.tif'
    iso3l   = country.lower()
    print(f'[Wealth Index] Trying WorldPop poverty index for {country} ...')
    for year in ('2010', '2015', '2020'):
        url_wp = (f'https://data.worldpop.org/GIS/Covariates/Global_2000_2020/'
                  f'{country}/POOR/{iso3l}_poor_{year}.tif')
        try:
            _stream(url_wp, dest_wp, label='[Poverty]')
            print(f'[Wealth Index] WorldPop poverty index ({year}) saved -> {dest_wp}')
            return
        except Exception as e:
            print(f'[Wealth Index] WorldPop poverty ({year}) not available: {e}')

    print(f'[Wealth Index] No automated source found for {country}. Upload manually.')
    print('  Meta RWI (93 countries + per-country additions): https://data.humdata.org/dataset/relative-wealth-index')
    print('  WorldPop poverty: https://hub.worldpop.org/project/categories?id=9')

# --- 12b. Temperature (Global Solar Atlas v2 - FULL GEOTIFF package) ---
# Global Solar Atlas has no public REST/API listing, but each country's data
# package is served as a static zip at a predictable URL once the country's
# display name is known:
#   https://api.globalsolaratlas.info/download/<Name>/<Name>_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF.zip
# (confirmed against the equivalent World-level resource links on energydata.info,
# which are served from the same api.globalsolaratlas.info host).
# Rather than unzip and keep only TEMP.tif, this downloads and keeps the WHOLE
# unzipped folder (GHI, DNI, DIF, GTI, PVOUT, TEMP, ELE, OPTA rasters + metadata),
# matching the folder you'd get from a manual download.
GSA_NAME_OVERRIDES = {
    'COD': 'Democratic-Republic-of-the-Congo',
    'COG': 'Republic-of-Congo',
    'CIV': 'Ivory-Coast',
    'TZA': 'Tanzania',
    'LAO': 'Laos',
    'BRN': 'Brunei',
    'MYS': 'Malaysia',
    'SGP': 'Singapore',
    'TLS': 'Timor-Leste',
    'SWZ': 'Eswatini',
    'GMB': 'Gambia',
    'MMR': 'Myanmar',
}

def _gsa_candidates(country):
    """Build a list of candidate Global Solar Atlas country-name spellings to try."""
    names = set()
    try:
        import pycountry
        c = pycountry.countries.get(alpha_3=country)
        if c:
            for attr in ('name', 'common_name', 'official_name'):
                n = getattr(c, attr, None)
                if n:
                    names.add(n)
    except Exception:
        pass
    if country in GSA_NAME_OVERRIDES:
        names.add(GSA_NAME_OVERRIDES[country])
    candidates = []
    for n in names:
        base = n.split(',')[0].strip()
        for variant in {base, n}:
            v_hyphen = variant.replace(' ', '-').replace("'", '').replace('.', '')
            v_nospace = variant.replace(' ', '').replace("'", '').replace('.', '')
            candidates.extend([v_hyphen, v_nospace])
    seen = []
    for c in candidates:
        if c and c not in seen:
            seen.append(c)
    return seen

def get_temperature_solaratlas(country):
    out = _mkdir(f'Data/{country}/Temperature')
    candidates = _gsa_candidates(country) or [country]
    last_status = None
    for name in candidates:
        url = (f'https://api.globalsolaratlas.info/download/{name}/'
               f'{name}_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF.zip')
        tmp_zip = f'/tmp/{country}_gsa.zip'
        try:
            print(f'[Solar Atlas] Trying Global Solar Atlas package for \'{name}\' ...')
            _stream(url, tmp_zip, label='[GSA]')
            folder_name = f'{name}_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF'
            dest_folder = f'{out}/{folder_name}'
            with zipfile.ZipFile(tmp_zip, 'r') as z:
                z.extractall(dest_folder)
            os.remove(tmp_zip)
            print(f'[Solar Atlas] Saved full GSA GEOTIFF folder -> {dest_folder}')
            print('[Solar Atlas]   (contains GHI, DNI, DIF, GTI, PVOUT, TEMP, ELE, OPTA rasters + metadata,')
            print('[Solar Atlas]    not just TEMP.tif - use whichever layers you need from the folder.)')
            return
        except requests.exceptions.HTTPError as e:
            last_status = e.response.status_code if e.response is not None else None
            print(f'[Solar Atlas]  \'{name}\' failed (HTTP {last_status})')
            continue
        except Exception as e:
            print(f'[Solar Atlas]  \'{name}\' failed ({e})')
            continue
    print(f'[Solar Atlas] Could not automatically download the GSA package for {country}.')
    print(f'  Tried spellings: {candidates}')
    if last_status == 404:
        print('  All attempts returned 404 - Global Solar Atlas likely uses a different spelling')
        print('  of the country name in its URL than any of the above (there is no public API to')
        print('  look this up directly). Manual fallback:')
    else:
        print('  Manual fallback:')
    print('  https://globalsolaratlas.info/download -> select country -> download the')
    print('  GIS_LTAym_YearlyMonthlyTotals GEOTIFF zip, then place the whole unzipped folder in:')
    print(f'  {out}/')

# --- 13. Blank CSV templates ---
SOCIO_PARAMS = [
    ('Country_name','','string','–'),('Country_code','','string','–'),
    ('Start_year','','int','–'),('End_year','','int','–'),
    ('Population_start_year','','float','People'),
    ('Population_end_year','','float','People'),
    ('Urban_start','','float','Ratio'),('Urban_end','','float','Ratio'),
    ('Elec_rate','','float','Ratio'),('rural_elec_rate','','float','Ratio'),
    ('urban_elec_rate','','float','Ratio'),
    ('Mort_COPD','','float','Deaths per 100k/yr'),
    ('Mort_IHD','','float','Deaths per 100k/yr'),
    ('Mort_LC','','float','Deaths per 100k/yr'),
    ('Mort_ALRI','','float','Deaths per 100k/yr'),
    ('Mort_STROKE','','float','Deaths per 100k/yr'),
    ('Morb_COPD','','float','Cases per 100k/yr'),
    ('Morb_IHD','','float','Cases per 100k/yr'),
    ('Morb_LC','','float','Cases per 100k/yr'),
    ('Morb_ALRI','','float','Cases per 100k/yr'),
    ('Morb_STROKE','','float','Cases per 100k/yr'),
    ('Rural_HHsize','','float','People/HH'),
    ('Urban_HHsize','','float','People/HH'),
    ('Meals_per_day','','float','Meals/person/day'),
    ('infra_weight','','float','–'),('NTL_weight','','float','–'),
    ('pop_weight','','float','–'),
    ('Minimum_wage','','float','USD/month'),
    ('COI_ALRI','','float','USD/case'),('COI_COPD','','float','USD/case'),
    ('COI_IHD','','float','USD/case'),('COI_LC','','float','USD/case'),
    ('COI_STROKE','','float','USD/case'),
    ('VSL','','float','USD/life'),
    ('Discount_rate','','float','Ratio 0-1'),
    ('Cost of carbon emissions','','float','USD/tonne CO2'),
    ('w_health','','float','–'),('w_environment','','float','–'),
    ('w_time','','float','–'),('w_costs','','float','–'),
    ('w_spillovers','','float','–'),
    ('Health_spillovers_parameter','','float','–'),
    ('fnrb','','float','Ratio 0-1'),
]

TECH_STOVES = [
    'Collected_Traditional_Biomass','Collected_Improved_Biomass',
    'Biomass_Forced_Draft','Traditional_Charcoal','Charcoal_ICS',
    'Pellets_Forced_Draft','Kerosene','LPG','Biogas','Electricity',
]
TECH_BASE = [
    ('name','','string','–'),('inv_cost','','float','USD'),
    ('tech_life','','int','Years'),('fuel_cost','','float','USD/kg'),
    ('energy_content','','float','MJ/kg'),('pm25','','float','24-h ug/m3'),
    ('efficiency','','float','Ratio 0-1'),
    ('time_of_collection','','float','Hours/day'),
    ('time_of_cooking','','float','Hours/day'),
    ('om_cost','','float','USD/year'),
    ('co2_intensity','','float','kg/GJ'),('ch4_intensity','','float','kg/GJ'),
    ('n2o_intensity','','float','kg/GJ'),('bc_intensity','','float','kg/GJ'),
    ('oc_intensity','','float','kg/GJ'),('epsilon','','float','–'),
]
GRID_TECHS = ['oil','natural_gas','biofuels_and_waste','Nuclear',
              'hydro','coal','wind','solar','geothermal']

def get_blank_csvs(country):
    import csv
    out = _mkdir(f'Data/{country}/Specifications')
    se  = f'{out}/{country}_socioeconomic.csv'
    with open(se,'w',newline='') as f:
        w = csv.writer(f)
        w.writerow(['Param','Value','data_type','Unit'])
        for row in SOCIO_PARAMS: w.writerow(row)
    print(f'[Specs] Socio-economic CSV -> {se}')
    te  = f'{out}/{country}_technoeconomic.csv'
    rows = []
    for stove in TECH_STOVES:
        for p in TECH_BASE: rows.append([stove]+list(p))
        rows.append([stove,'current_share_urban','','float','Ratio 0-1'])
        rows.append([stove,'current_share_rural','','float','Ratio 0-1'])
        if stove in ['Collected_Traditional_Biomass','Kerosene']:
            rows.append([stove,'is_base','FALSE','bool','–'])
        if 'Biomass' in stove or 'Pellets' in stove:
            rows.append([stove,'draft_type','','string','Natural or Forced'])
            rows.append([stove,'collected_fuel','','bool','True/False'])
        if stove == 'LPG':
            rows.append([stove,'diesel_cost','','float','USD/litre'])
        if stove == 'Electricity':
            for t in GRID_TECHS:
                rows.append([stove,f'capacity_{t}','','float','GW'])
                rows.append([stove,f'generation_{t}','','float','PJ'])
    with open(te,'w',newline='') as f:
        w = csv.writer(f)
        w.writerow(['Fuel','Param','Value','data_type','Unit'])
        for row in rows: w.writerow(row)
    print(f'[Specs] Techno-economic CSV -> {te}')

# --- 14. Specification templates (blank, generic - Technoeconomic & Prep file) ---
# These are NOT downloaded from anywhere or generated from live data - they are fixed, blank
# workbook templates (Read Me + blank table + Parameter Legend tabs) that apply to any country.
# Embedded as base64 so they save with no internet dependency.
_TECHNOECON_TEMPLATE_B64 = "UEsDBBQAAAAIAAl1+1xGx01IlQAAAM0AAAAQAAAAZG9jUHJvcHMvYXBwLnhtbE3PTQvCMAwG4L9SdreZih6kDkQ9ip68zy51hbYpbYT67+0EP255ecgboi6JIia2mEXxLuRtMzLHDUDWI/o+y8qhiqHke64x3YGMsRoPpB8eA8OibdeAhTEMOMzit7Dp1C5GZ3XPlkJ3sjpRJsPiWDQ6sScfq9wcChDneiU+ixNLOZcrBf+LU8sVU57mym/8ZAW/B7oXUEsDBBQAAAAIAAl1+1wk+zMd7wAAACsCAAARAAAAZG9jUHJvcHMvY29yZS54bWzNksFqwzAMhl9l+J4oSUs7TOrLxk4tDFbY2M3IamsWJ8bWSPr2c7I2ZWwPsKOl358+gWr0ErtAz6HzFNhSvBtc00aJfiNOzF4CRDyR0zFPiTY1D11wmtMzHMFr/NBHgqooVuCItdGsYQRmfiYKVRuUGEhzFy54gzPef4ZmghkEashRyxHKvAShxon+PDQ13AAjjCm4+F0gMxOn6p/YqQPikhyinVN93+f9YsqlHUp4221fpnUz20bWLVL6Fa3ks6eNuE5+XTw87p+EqopqlRXrrFrvy6VcFrK8fx9df/jdhF1n7MH+Y+OroKrh112oL1BLAwQUAAAACAAJdftcmVycIxAGAACcJwAAEwAAAHhsL3RoZW1lL3RoZW1lMS54bWztWltz2jgUfu+v0Hhn9m0LxjaBtrQTc2l227SZhO1OH4URWI1seWSRhH+/RzYQy5YN7ZJNups8BCzp+85FR+foOHnz7i5i6IaIlPJ4YNkv29a7ty/e4FcyJBFBMBmnr/DACqVMXrVaaQDDOH3JExLD3IKLCEt4FMvWXOBbGi8j1uq0291WhGlsoRhHZGB9XixoQNBUUVpvXyC05R8z+BXLVI1lowETV0EmuYi08vlsxfza3j5lz+k6HTKBbjAbWCB/zm+n5E5aiOFUwsTAamc/VmvH0dJIgILJfZQFukn2o9MVCDINOzqdWM52fPbE7Z+Mytp0NG0a4OPxeDi2y9KLcBwE4FG7nsKd9Gy/pEEJtKNp0GTY9tqukaaqjVNP0/d93+ubaJwKjVtP02t33dOOicat0HgNvvFPh8Ouicar0HTraSYn/a5rpOkWaEJG4+t6EhW15UDTIABYcHbWzNIDll4p+nWUGtkdu91BXPBY7jmJEf7GxQTWadIZljRGcp2QBQ4AN8TRTFB8r0G2iuDCktJckNbPKbVQGgiayIH1R4Ihxdyv/fWXu8mkM3qdfTrOa5R/aasBp+27m8+T/HPo5J+nk9dNQs5wvCwJ8fsjW2GHJ247E3I6HGdCfM/29pGlJTLP7/kK6048Zx9WlrBdz8/knoxyI7vd9lh99k9HbiPXqcCzIteURiRFn8gtuuQROLVJDTITPwidhphqUBwCpAkxlqGG+LTGrBHgE323vgjI342I96tvmj1XoVhJ2oT4EEYa4pxz5nPRbPsHpUbR9lW83KOXWBUBlxjfNKo1LMXWeJXA8a2cPB0TEs2UCwZBhpckJhKpOX5NSBP+K6Xa/pzTQPCULyT6SpGPabMjp3QmzegzGsFGrxt1h2jSPHr+BfmcNQockRsdAmcbs0YhhGm78B6vJI6arcIRK0I+Yhk2GnK1FoG2camEYFoSxtF4TtK0EfxZrDWTPmDI7M2Rdc7WkQ4Rkl43Qj5izouQEb8ehjhKmu2icVgE/Z5ew0nB6ILLZv24fobVM2wsjvdH1BdK5A8mpz/pMjQHo5pZCb2EVmqfqoc0PqgeMgoF8bkePuV6eAo3lsa8UK6CewH/0do3wqv4gsA5fy59z6XvufQ9odK3NyN9Z8HTi1veRm5bxPuuMdrXNC4oY1dyzcjHVK+TKdg5n8Ds/Wg+nvHt+tkkhK+aWS0jFpBLgbNBJLj8i8rwKsQJ6GRbJQnLVNNlN4oSnkIbbulT9UqV1+WvuSi4PFvk6a+hdD4sz/k8X+e0zQszQ7dyS+q2lL61JjhK9LHMcE4eyww7ZzySHbZ3oB01+/ZdduQjpTBTl0O4GkK+A226ndw6OJ6YkbkK01KQb8P56cV4GuI52QS5fZhXbefY0dH758FRsKPvPJYdx4jyoiHuoYaYz8NDh3l7X5hnlcZQNBRtbKwkLEa3YLjX8SwU4GRgLaAHg69RAvJSVWAxW8YDK5CifEyMRehw55dcX+PRkuPbpmW1bq8pdxltIlI5wmmYE2eryt5lscFVHc9VW/Kwvmo9tBVOz/5ZrcifDBFOFgsSSGOUF6ZKovMZU77nK0nEVTi/RTO2EpcYvOPmx3FOU7gSdrYPAjK5uzmpemUxZ6by3y0MCSxbiFkS4k1d7dXnm5yueiJ2+pd3wWDy/XDJRw/lO+df9F1Drn723eP6bpM7SEycecURAXRFAiOVHAYWFzLkUO6SkAYTAc2UyUTwAoJkphyAmPoLvfIMuSkVzq0+OX9FLIOGTl7SJRIUirAMBSEXcuPv75Nqd4zX+iyBbYRUMmTVF8pDicE9M3JD2FQl867aJguF2+JUzbsaviZgS8N6bp0tJ//bXtQ9tBc9RvOjmeAes4dzm3q4wkWs/1jWHvky3zlw2zreA17mEyxDpH7BfYqKgBGrYr66r0/5JZw7tHvxgSCb/NbbpPbd4Ax81KtapWQrET9LB3wfkgZjjFv0NF+PFGKtprGtxtoxDHmAWPMMoWY434dFmhoz1YusOY0Kb0HVQOU/29QNaPYNNByRBV4xmbY2o+ROCjzc/u8NsMLEjuHti78BUEsDBBQAAAAIAAl1+1zoPR9tYwQAABAMAAAYAAAAeGwvd29ya3NoZWV0cy9zaGVldDEueG1snVfbTiM5EP2VUh5gV8o93AZCpFmGmUFiBzRc9jFyuitpC7fda7sT8vdz7E4CCOggHpJ02+W6nDpV5QwXxj64jNnTY660O21k3hfHnY5LMs6Fa5uCNXamxubC49XOOq6wLNJ4KFedfrd70MmF1I3RMK5d29HQlF5JzdeWXJnnwi7/YWUWp41eY73wW84yHxY6o2EhZnzD/q64tnjrbLSkMmftpNFkeXra+No7/trbDweixL3khXv2TC4zix9WppewjEC6DQrBTYx5CNsXaViCPCtOfFAq8DPnM1Yq6IZn/6/MNDZehIPPn9f2vkc4EN5EOD4z6j+Z+uy0cdSglKeiVP63WfzkVYjR5cQoF79pUcn2enAwKZ03+eo0XMilrn7F4wqbTnUwWv0mvBgNrVmQjVJR+9Fay8YegkyCRIwpCmJV6pCQG2+xK6HQj85Mqb1d0i0nmTbKzJa0I/LihL6XrOim4EROZSICVI5aEMsLJTwPOx5eBRWdBB94s3Gpv92lfnSpH10KjHmlY1DpGOy9r2PwTMfrsC50UXpKARWBtOQzpit9c3t1f04JqCD1rMWaLaLNTcrqGKvON6lgGzmuE24S59IF3rkmCZ3CEWtZeyodiBq18pyB3Eof+ScEg/g04PfXRBrUDzQkmbCJEapJD2yNg/EmXV7/gBWfCRxrEkRnApKRmVYm0i//btfgvLcd571tOO9vx3m/FufbjB2Te0kTYRmoRF611lvgzlyokh1yITzqX3oyeiOHelCKtFntABGkzLbpDtpD8nKkB6WKHIB8TSqLljctpJerHIu5kEpMFG+yvdY74ZCbmGTFaR2eB9vxPNiG5+F2HYdRx+AdPH9CjzfgWAhbuk3vqvH7aLvNo7ochpZ/7AqR8GkDPd2xnXNjRNRr0/eQFKkjoPche4BVlblG5mi36hicGG1ymYzXnWEXqfTZ6/yHPNXB/2V7GF8+E0a/ItHutbAiZ892fMkz1ukuCTAV1biMnFkEVrJIMoqClLMIhS+9qxjmlwVXfaDU0tcFEjr61p7c/UwogzadPxbG+hrwEdSO8icXN1eDnZk/GfsXgu3EzWO4gWBI7Kon1obzkRHT21YYvQ9MhV6/tjR+Gc+uztHBB0zUTo33cG/RN6N3QY8UrSnUQmDEODCCAOUd+LCqC1cL5N72Ztvb+5x/lyzmDDZXRTpRQj+gRtWS5JSCz7bEc2rYhTBEUajlSaSBoFkYg7FvvxhgExdG3dSaPAY8w4UK15JHTCeNEqIu4QwadGjmmhSshy4LS9F0LQofGDm92pnzPgq4j1EisIPROQ5V8BRRqAXqPIv2rX246FDeKeI6fxrCTXpDFK05TC8zPaZsmVrTJGeUsE00P502YSfOrzzM+2rqG4kvLTwGmBrHOf+rTACcjYM/XBbcGN1lvBDO85sAdp7d/sJd+V9hZxLTVvEUMHXbh0DNVlBWL94U8d43MR4wV3dFXNnZBgHsTw0qavUS7pibPwGjP1BLAwQUAAAACAAJdftcUf+wwF0VAACv5wAAGAAAAHhsL3dvcmtzaGVldHMvc2hlZXQyLnhtbK2dUXPbyLlE/wpLqbpvCTkgCZJ7bVfFsr3ZZJNyrbPJowomIQllitAFoVWcX39BWdRMNvj6zGLwkrVsdUM6oaVWG4N+9Vg3X463ZdlO/nW3PxxfX9y27f130+lxe1veFcc/1PflofuT67q5K9ruzeZmerxvymL3JLrbT7PZLJ/eFdXh4s2rp9/72Lx5VT+0++pQfmwmx4e7u6L5+rbc14+vL9zF+Td+qm5u29NvTN+8ui9uyk9l+/P9x6Z7a/risqvuysOxqg+Tprx+ffFH9937bJadFE/v8o+qfDwGv56cPpfPdf3l9MYPu9cXs4uT96GcfP10v6+erjZp6/sfy+v2stzvO8fsYlJs2+qX8mP3bq8vPtdtW9+d/rz7ONui7X7ruqn/XR6erlnuy+59u4/m/r/e+ZvJs+npk/y/54/44uUTOn1Q4a/PH/mHJ7Idqc/Fsbys9/+sdu3t64v1xWRXXhcP+/an+vFP5TOt5clvW++PT/87efz2vll+Mdk+HLuP5lncfQR31eHbf4t/PVMOBZkhyJ4F2a8EbmEI5s+C+a8FM0OweBYsfi2wPofls+DpU59++9yfwL0r2uLNq6Z+nDSn9+7cTr94ov90mY5XdTi9Cj+1TfenVadr33x4KPevpm3ndHp7un1WvdWqj0VT3PXILrXsH8X+oeyRvdOyXfeJXbVf7/uk77X050PV/qdq2gF6oZS9UMqebJaGTfcyPL3Wy93V35tiV51e9MX+6m1Vd18Rjn34vtnlht2huOv7XC6/qVZPqtOXDw9I+x3bpjrc9NHRuv/53Tpz7n8FoPkLoLl0+s2AtF11+OVqWx/bPkhzE5L2vN7XRZ/hey37+dM7wWfxwmcxLh9t15bb26t9dd37KlqYgLRpdejFo0Vfy6I5CkDLF0DLcQFpu+vuy5r5ClqagMDUegVpWfcKmn65EYzyF0b5uIy0XXkom5uvHaVDW/b+X3+Zm6C0swlKy/76Z81p9cJpNS4nbXd/ly376KxMOtrPpKNl2eL3t5OHm+ndXCBavyBaj4tI25XX19W2Kg/br32g1iYo7WqC0rKfiu7Tmcx+7wSnzQunzbictF3bBfir+rr7O7d/zs59vDYmL+1u8tKy2/qhOU53xVfBy818npyNSwz8PLL6S3/IuXx26AMG5iYx0EUhCyK4GxmZ9qvvzG99z8peVNrURqV1p+9+p5CgSPkY7nRg/e2ktN+2zq6q07e/Y9X2fulydiQHa5uX1n25mX7/ZwXLR3I3ciYHv+3tAmDZ0RysbVhah7B8PncjB3TwO2Q1wLJjOljbsLQOYfms7kYO6+D3eQus7MQOzjYrrUNWPrO7kUM7+NXEyg7t4Gyz0jpk5XO7Gzm4g195f6z2/RnL2ekdTG1MWsdli/Pp3Y0c38GvOl6d2tZeUHZ6B9PPdd1XLb4H2d9/+vn99MMff/z0XqHyAd6NnODBb/vQNN2Py1fH26Iprx6az0X/68sO8XAB+/WldTE/9mQ+xmcjx3jw+09szUNT9L02LjM7ysMFTGygi8Lmo3zmIoviH+7um/qX7heKmY7OZk1sh3hwFEWxFvIXryzo0mNDfBQjbaaa4kz06QPDO+h0WZz56J7FRvcoRNpMlsWZndnB1qiLQUV9ceYTexab2KMYQaZWfXFmR3WyNV9HWoeVceazehab1aMwaTOujDM7qoO3zUrrqDXOfFTPYqN6FCptZrXGmZ3PwdEGpHVRxXHmQ3oWG9KjKEFCl8VxZod08LVZaV1UGvAxPYuN6VGotFlcd5zZcR38bWRaF9OFZj6uZ7FxPQpZbN1udseZndPB3OaVXrfPfU6fx+b0GF5gJorjuR3MwdT+d3etiyiO5z6Xz2Mr9ihM2gxb47kd0MHahqV11MHMfTqfj5nOwQwr47kd0cHaJpXWr8+DW17GDOlghn3xXNz3MvTGl7Ryfe6j+nzMqA5mVBbP7bQOzjaotGJ97sP6fMywDmbUFM/tqA7ONqi0Vn3uo/o8lz3LM5XJh7rZlrvJu6a47vuA3s51NLYalrkd0sHRblhAGHEvno/oc51io+loG3kXnh3NwdV+8Wid7lbmPpTPdV6NhgNxXLUqczuFg63RqoCKWpW5z99znU6j6UA4Vn3K3M7cZGu+drQO+5SFD9wLHUtjAYENNykLO3GDt0kJdNSkLHzcXug4Gg1J21gdysLO1+Boo9G6qA5l4UP2QofQaD7aRrcnCztbg69NSeti2pOFz9cLnT+jIUH9HdWbLOx4Df42LK2L6QEWwf3lOoNGw4Iby7kxWYj7ywfma9BFkfIZe6FDaDQpSNd2V7KwgzWY2oT4FnPoShY+Wy90Eo0GpG2wJVnYIRusbUxaRz9/LHzCXoyTsMEG+5GFHbPB2makdcjIB+3FOEEbbLAZWdhhG6xtRlqHjHzcXowTt8GGOpGFnbjB2UakdYRo6QP3cpzADTbUhiztuA3O9rEgrUNEPm4vx4nbYLM76ayDlpdLO3SDr92JgPBvRXu6UWhSN8+fmqLlw/dynPANNtuXru66/wzt5dIO4OBt3ToHsqhb55Y+gC/nsmT7WHafYRsBCqpto2Rb2pEbHMULSgu5ZFv6xL3UqTSaDp26tEu2pR21wdX+moQ3nig4wWlOHUij4WgbWbItxTlObWuUbKDCo64+YS91Do2mAzFYHnK10zXZmq8drcOSbenj9VKH0GhAcHcJlmxLO1+Dt01J66hkW/p8vdQpNBqStrFKtqUdq8HRRqN1USXb0mfrpY6g0Xy0jS7ZlnayBl+bUvp93rlP17kOobGQwCauZMvtjA3+9ony9IOauc/Zuc6j0bC0TUTJltthG8xtUloXRcpn7Fwn0WhS2kaUbLkdrsHUJoQ3d1PJlvtsneskGg0IzmRSyZbbIRusbUxpt43kPmHn4yRssMGSLbdjNljbjNLuGMl90M7HCdpggyVbbodtsLYZpd0skgePTRknboMNlWy5nbjB2UaU1mfnPnDn4wRusKGSLbfjNjjbiNLq7NzH7XycuA02umTL7dANvnYnAsLfUrLlPnzn44RvsOGSLbcDOHhbJRvIokq2lQ/gq5ks2cLjlZe3RbOte09Gvl3poGuVbCs7coOj/YICIZdsK5+4VzqVRtPRNqpkW9lRG1ztBznhI1EUHB+yV1D3xsLRNrJkW9kxG2yNkg1UVLKtfMJe6RwaTQdisCrZVna6JlvztYOPI4SHpPl4vdIhNBqQtuGSbWXna/C2KWkdPknO5+uVTqHRkLSN+Qw5O1aDo41G66JKtpXP1isdQaP5aBtdsq3sZA2+NiWtiynZVsHzCHUIjYakbeJKtpV4LuHQBxNqXUx1tPI5ewXP/4iFFXtc0izZVnbYBnObVPpZyZXP2CtIorGktI0o2VZ2uAZTmxDfqQ0l29pn67VOorGAwAZLtrUdssHafsRl2r0ja5+w1+MkbLDBkm1tx2ywthmlnYlc+6C9Hidogw2WbGs7bIO1zSjtNOTax+31OHEbbKhkW9uJG5xtRGl99toH7vU4gRtsqGRb23EbnG1EaXX22sft9ThxG2zEg9/WduIGU5uO1nEfsvZxez1O3AabyIeYre3cDRewWaXn7rXP3etxcjfYRD67bG0Hb7iATSv9aSXr4JHga9lEnglNfrj81EsJ7tM2Gsi1ePz30EobhBF/43zIXsMDSYiKlqvmcW2na3C1XzCYrgWUjQ/WG3jqCEABuWwcN3aYBlujcQQVNY4bH6U38JARogI5VzWNGzs+k635VHh+cLcs0TY+P2/gmSIEBo45YsO4sYMzeNt0tI4axo0Pzht4jAjB0XKrWdzYQRkcbSRaF9Usbnxa3sBDQ4gL1NKyUdzYGRl8bTpaF7VK4XPyBh4UQnDgvuq4HQo7KoO/DUnrYvqxjQ/LGx0mEZKWRzSIGzsgg7lNSOuiCPmAvIFn+REhuN3Dbg43dhgGU5sMPh+EmsONz8IbeHIfgdFybAw3digGaxtP2pnFTTCJk5aIQY5N4Ubs3wwdwEk7rOhm4fxNWjQmPQ9HzNTczeC9m7S+2c2CtZtZWkwmPa5FzMTIDXgLQGlls5tlAaC0uEx6nIiYiVUb8BaAEndtZsGwzUwfTPxL2dTH7qeCfjjDTiOedf1Qhp5HJGXEKMQs2LCZ6YypuQw/h3jW9rMZulYDQt1VuFmwVjPTqVJjSTiBeBb3cxl0BpFk1Fa4WbBMM9NZUnOB/KrKirO4n8vAYExC7CvcLBiimekwqdGknjo8O/TzGRiPSUiNhZsF8zMzHSY1nmHnDc+6figDQzEJozoLNwvWZmY6SmoyKScNz+p+PkM3ZUAY01q4cBwShhMlnhGWINUU5OAtyFHGIMM1SB0lNaOE6Ue1/Th4/DF9/TGcf4RxRI0mdetRjT0OXntMnXsM9x5hDVHTSR13VOuOg+cdU/cdw4FHmD/UdFLXHNWc4+A9x9RBx3DREQYPNZ3E+Ua13zh4wDF1wTGccISFQw1Ha/GHcTXYOHixMXWyMdxshF1DDQeishhoVAuNgyca0zcaw5FGWDLUZKBBjpwZVPOMcAlBKX1g3YULjbBcqDn9ljlG894cp/YYBw8yjrDI6IJJRpfpo4I/fvy+FxEsHJqNlxhbJE/ReIEy4q9YMLfoYJfQRDL8PKAT84rkK0a/U44EumBc0cEEoUkk4RCgE0uKZGz1XCDDnisYU3SwOWgiSTj558RwIhrbr5LEw38umE90sDFoUkk97ufEXiK5CzRpJ/5csJjoYFHQJDPsjJ8Tm4jkKXiMcMzPBduIDlYETShaBp2WWEMkZ4FGC+O+LQcBGDYDTTJjnOZzYgORriAIpR/oc8EMooOJQJNQ8hE+J/YOyV7gST/F54LJQweLgCaewef2nJg1JFuBJfnonguGDR3s/1lUQMY1n5gyJHObDQjxB+5gytDB1p8JJvGEnhOzhWQuwCTeNxEsFzqY9jPBJB7Lc2KlkMwFmMRiOBgqdLDkZ4JJO4vnxCYheQsuiZVwMEvoYLjP5JJ2AM+JCULyFlwSy+BghdDBTp/JRct2VXlUPzKJwUGyFlh4GWVftU0p0QQxGGb6TDRaFlvnieFBuoRANEIeDsYHHQz1mYhGOWXnxPIgXUIgSj9o54IJQjfXR+3et7fFoffpY28dTP2ZbZ7YHCRP0eaBMqLNC7YHHcz0SSzDz9k5MThIvuIlk3LUzgWDgw5m+RQVkOpWTywMkrHV6oEMW71gYtDBJJ/EknDSzol1QTQ2Xy0g5GYvGBd0MMMnyaQetXNiVpDcBZ6003YuWBV0sLsn6Qw7a+fEeiB5CiYjHLdzwYKgg5k9CSblsJ0Tk4HkLPCkn7dzwWqgg4k9SWeM03ZOzAbSFQSl9AN3LpgOdLCwJylpaUzbJyYDyV4g0sI4REE8hoE9iWjweTsnlgLJVqBJPnLngr1AB7N6kkzigTsnVgLJXPBJO3PngqFAB4N6Ek7iiTsn9gHJXMBJPHQXTAQ6mNJTcEDK7Z9YBiRzG07iNqALxgEdrOhJOIlH7cQoIHkLNomVcTAF6GAxT7JJPGUn5v/IW7BJbI2D9T8HY3mSjZZSEyim/8haoOGbJ7AJDOb/HEzlSTywSBLZBor9P7qEwDRCbg6GAB2s5klMWhrbCIohQLqEwKSFcZiC4Pw8nmc1gm+r+qY49lPS+dQsBMUCIHmKQhCUEYVgMAPoYCxPUdFK2QeK1T/yFS8YzMoSShCTYSZPQYEbKmQdKAb/yNiqA0GGdWAw9udgFk9RgQQr20Ax8IfG9muF76TQfVcw8OdgAk+AAWVEGSgW/cjdpgNCLAODQT8Hq3cKjlaaXaCY7iNPgUQL47rAPMjFOW5km1ygQdZVYC5CMTgLOloY9b06mO5zsHCn4GhlZBMoRvvoCgKSFkbVXMF2n4OJOwVJK2OKQDHZR/aCkBbGEQqSMQzcKUJaqXpAMdRHtoIM32RBPWAw1+dg1k6B0UquAcVIH5kLPFqIP5QHO30O9uwUG7i/AltAsc5H5oKNFjKbIBjDkp1iAzdVYAkohvnIXLBJrI+DOT4Hq3WKDczkUc8lFvjIW6BJLI+DAT4He3UCDSixAhTre+RtowEhowni8fNSnXmf0imlNF2e6/383joYujOrCbG5R56imgBlRDURbO85GKpDNHQsTvQTYnSPfMXLBtetJZkgHsNEHZKBjCxLCrG4R8ZWSQEyLCmCxT0Hu3SIBqKrbCrEzB4a268afEgb/DAeTO05mKZDOjD6wXWFmNsjd4FIC7GuCMb2HMzSISEtNzsLMbBHnoKLFsZ1FsHInoMZOoQDz6LQxYWY1SNngWiE246DZT0H83NIKPY8nmwvxKweXUGQGuFIXrCs52CGDklpeUyFIbb1yF5g0sIoTMG+noMZOsIEctVjiE09srXxgDCmxwiW9RwM0CEdOLAX+a/EYlmPLiFIaWHUV6ZgYM/BDh2igme7Rf5LsRjYo0sIVCO0z8HOnoM9OkQFD3qrq8O22nXf58qr62Lb1k0/KZGzhw7tkTDiB7RgbM/BKB1y0vKbpto9fXWSlETkHrq1R8IYSkHmhnE6pAS3aBT3xUl7dft119T9jETmHrq4R8Lv/ynxBIEb9ugQD/TRZzzHev/rbytnPCJ6Dx3ZIyHgCSI3DNAhHqikz3geq8Oun45I3UNH9UgIdIK4DTN0SAfuaT7TuSnr9rZs7qxvaiJvwyUEIy0ERkHQhkU6ZAS3Np8ZGdOOl06M7JG5oKOFmk4ws+dgm47ogPyFTl31wxFbe+RtwwEhwAniNQzUIRyI12c4h6I9Bcar/n8EuHRieo+uISBpIUAKgjXs1CEkCNZnSH972O7/6yejMyCRqofO75EQAAVxGlbrEBDE6TOgz1V9qmePV8Vhd/VYHNv+/lrM8tGlBCstBFZBpIYNO2QFkfrU0J5+GjqIuCiW+chfANLCj/Lfy4JVPgf7dQhIywNAdmAUq3zkLwBpIQAK8jTM1yEgLQ8AmZFRbPKRveCjhcAnCNQwYod8tDzgA6FR7PPRRQQlLQRKQbCGLTukpOUBJTM2ioE+shd8tBD4BKEa9uyQj5YHfMzgKFL10JE+Eko8WTDRl8GYHeAheYAHomMmpvroKiYmEgImF2BKytckDzCJ8JiJsT66gkCkhYAoCxAlpWuSB4ji4mMmlvvoYoKWFvbTmh5vy7J9V7TFm1fFQ1t/qPZt2Uya8vr0Beq7J9fpm1f3xU3516K5qQ7Hyb687i4w+8PpH8yb6ub25Y22vu++wF1MPtdtW989/fK2LHZlc3qH7s+v67o9v9F5Th/r5svT5d/8P1BLAwQUAAAACAAJdftcUt6roxkJAADmJgAAGAAAAHhsL3dvcmtzaGVldHMvc2hlZXQzLnhtbLVa71PbOhb9VzTZmS7MAMbmZ1tgpgXa9960lCnt68eMYsuxFllyJRnI/vV7r+zYoXVu+jzbL5A4PrJ0dHXuPZLPHo29d4UQnj2VSrvzSeF99SqKXFqIkrs9UwkNv+TGltzDVzuPXGUFzwKoVFGyv38clVzqycVZuHZrL85M7ZXU4tYyV5clt4u3QpnH80k8WV74LOeFxwvRxVnF5+JO+K/VrYVvUddKJkuhnTSaWZGfT97Er66SEwSEO/6W4tGtfGY4lJkx9/jlz+x8sj/BprVgi7tKyfAw5k31QeT+UigFDSYTxlMvH8Qt3HY+mRnvTYm/Qzc993Apt+a/QodnCiXgXuhM9dPNTSNtozjG722HJ914sFOrn5c9fxeIBaJm3IlLo77JzBfnk9MJy0TOa+U/m8c/REvWEbaXGuXCX/bY3JscTlhaO+hNC4YelFI3//lTS/IKIN5fA0haQPIj4HgN4KAFHPz0hHWPOGwRh4GaZiyBiCvu+cWZNY/M4t3QHH4IbAYUjF9qDKo7b+FXCTh/ccstL88iD03hhShtYW9pWAbPmvpFJQaglzT0q5Z+AHVFo66ES62sMHyegyMYcDfqpBt1Elo7WtPaLJ1K7XFp+MXQ4Bv08Rp0rgwfGsIlDbufR+//Ghp5AztdA3ureHrPUm5nsHJEKV1YzzmsGGNZJSybyzn/j6mVYCZneS0UE1rY+YKlRoNWiGyPYOygY+yAZCzlFU+BrekL5V97kRbaKDNfvJj710MEHowjkIa9/zbE3gHJ3p8adEgpkbE5ssKD/iwHg4T9PB62VSwya3aYM4rbHViROtsBuPGFAKlRO0As/jUS/mjua8vVdM7dDrupUyUQMpMGJ8JNuc6mj9x5sc2kZtAAAqALHCYJ1dDK0I+5lRms7idqpg67mTqkZ6o4pIP7cNzc0LC1wX1ITs9H4QtMMP/3uD7q2Dqi2TIJzdbROLZo2Fq2jki2LhsRyKR5ktlvIO24I+14A2lSp9ADnYpp8+wh5o7HMUfDXvzrNInjIc25Oia5e9dwxNPU1NpLPWdQkbHC1E4URmVAHr9/tiYzqN50xrTxrB0vgh6lL5rlGgDNTa8ZtJLhCm8byGXaKo1xAOcqrVW44Cj6Tzr6TzbQr/AxIpviBA9Rf0JyCCXeEOqSRn35/PU6evfmw931EPknJPnfCoHiGQQwBKV0bM7xEtA2W4Tr/VRs4aO2meUthkPFWNu04EjyVujCNsXjacfjKc1jba3QfuoKbsXUoo4PkXk6Lo5p2GcMB7a/Gw+ReUqrQNNtFrqNyzz0vOfPQTRiqPoCWIYC8kFEgXPukGcJOmElGggITnMfbuzSH8Xqy47Vl/+A1drOuB5i9eU4VmkYyerLf8Zq6PlvZzXe7wv1fZLXTApwUFNUlMF6fX8coRtwX++uIrB+dqjQv2qxaylF8QMmm56H9BSa2mnk0hsmnAfSvAgC4C3XrjLWR5lQ4AYDl00LgfRA95bYm++xD7fvd9g1KoNR29iQq6tKLVgFQu1JjY1XjFFM82157td5nLcteB1tDpKAng/yTQNvmnqSQW4CY5uKbJD3mOT9DURj6D3D3oc8BwVpyZ0LwRmCF6rVx0KmBeN5DrkEfylnYDcxaYkc8peE7L5gmAGXZQbNa2+9Ytp79a0P8jrSem3AkcIQ0wbsS1P2r9ISQlI0MvAKozmvu0KrDV4I8Ew+QJHUJbi+EPMoNMv67FcKs7i3aTHt05qHTNuHDDI80pttwH38K7ofCvirmDZoNyIUSCYUTbC8Q/+32QNXtVjyHHhC+biXoKeWlyRXvVGKaackKgftDWaneKRJ2oAjqteYNkrXT5VxtRW7VoBGaifApQqufLHNKtzCET6oqw6ph4e9MGwiUPdvx24/JntH/UpGyezrLY7ZC2rKuln97ZP22CcNkmrF91piNAcZaZuNQKsz2RrZ4HTJGenNWEy7MWxqfYYbacU24DDDrYlc2o19Fp5LhXk/JSKVXWucmv3AX6AKK1rPsMroCvl1tW5IdjnQ/2hMRta8ce/dYtq89fsgv7iXE480chtwt4MGOKZd3But6x/3Tvp9ndyacmhD58etl+V2S1+IZMKCaIe7wo8/WGuS+d62xbRvw5ZDdBOmOaZN2HqyaRwlPLR1+1grLyslITp5hf8DX0gUikHL1sp0hJIN7rAix6ss2G272HWVSIPEew5Sn2OB4+qZk1nIiI2Y0CrS27qY9nVSP6wXkZFubgMORGSQW9rJfa0gYjXmv0p6iMut5dKPYAhQG5eQwrdXiuC24ggC08k3yVlv2mLatUk3xfkcpIz2XOt2EzbA6O2EmDZrH7m9d1DALncVwB30Sx43F8BlcNTVxp316XCrdacRnk7Vbvd7bbY7x2bC8cJSLUoDldweu8HjJQWJEDvMDKZEVPLLbg/mS58Kp2+bMnuHhYGxLUyZiuv77YCBZljYRCbjPOkdYUI7Qp2YDccYIz3hBtz6gwzaD96A+YOoZb9p9zLprV1CWzsN9dIgX2NN3QYgob4JbeVuoKfLtb8S4eIJOINIhPCWuGUBlGUoz4LbZfh+0ndfPv193YQxM7WvatoZJyuHZ7SDM5tOz8Yen409P6P92yc75xoSz+86Qkt6c5bQ5syUa/NSMtKVbcBhcbuAqBikjTZmballqmV1hXsB+JoCzDzXqfg5L5Ek9a4soV1ZVSZHgwyNtGQbcMnhbsHqeVQeDHJEu7I/Og/13GAtYwtVv6NnYNtL6sxgYnhmwJDnxtsxWVbQEL1se3OV0OYK9WOqZD4sfbRJkoN7CZcbUBh5bpBW2lmB2W1sURt8kJux30CaeF4IzQQwLFADtRCZg6KzUjyF+0nGequU0FYJHzg1+bT1aT+9e9BSN9IfbcCBqFsXZXxIYq+SDTbpAXibQ8pAxlDgoBnGVyw/VOM6c0sD2u1ro/Q9d6y9GW28azj/0ob1lLBZbTOhScZ7i5TQFqlnPJRkg3SPdEgbcDTdtEf6Vbqbt5xUf0YQ2OxPFAY5jFZe7uG1N++kwun54YUufAMMquK51I4pkUM39/dOgGXbvPjUfPGmCu8RNW9ehY8gNOB98Qb4PTfGL7/gy0Xdq20X/wNQSwMEFAAAAAgACXX7XBA+UsMsAwAAIxEAAA0AAAB4bC9zdHlsZXMueG1s3VjtbpswFH0VxAOMBFoaphCpRYs0aZsqrT/21wkmsWQwM6ZL9vTztQmQxrdL1q2qBoqwfX3uOb6+/lDmjdpz+nVLqfJ2Ja+a1N8qVb8Pgma9pSVp3omaVtpSCFkSpatyEzS1pCRvAFTyIJxM4qAkrPIX86otl6VqvLVoK5X6Ez9YzAtRDS03vm3QXUlJvUfCUz8jnK0kM31JyfjeNofQsBZcSE9pKTT1p9DS/LTmqa2Bys5PySohoTGwDE95biUjHOyrzsNAIDcrrXYyXV59uJkdscTnODzW9Dfpwxc5XJrncoV978ml9E6k+TTaA+O8z4Qr3zYs5jVRispqqSsGYxpPTF5XftjXOhU2kuyn4bV/NqARnOVAucncIQ9G0Bc61TEPswx1aj46HCshcyr7gIT+oWkx57RQGi7ZZgtfJWqIs1BKlLqQM7IRFTHROiDGSM8s6tRXW7Moj1Libgmv0QZdO44zEaavkXMmQPc86D4TYTuPBtYVdLzWlPOv4ORb0Qdtql3tCs/uOx9z2HI8yLZDUUe6K1o3tgJEY2/W98ht8kduvZo9CnXX6hFUpv69FYreS1qwnanvip4f8z7FvZO65vtbzjZVSe3YzyZczMkB5z1SqdgaFqmeHd/7IUn9QHeqW8DBrsDFhW9ZXPSWxV0N4sKxuOk/ELfWVSp/KylG4nUB/zN5fP1a3qNX1f56mRR0W9Jo3zva9fpWD47k1P8CFzQ+0HmrlnHFqq62ZXlOq5PNT7tXZKVvgEf+df+cFqTl6qE3pv5Q/kxz1pZJ3+seQtD1Gsqf4LSwFyiz22suVuV0R/Osq+rt/+jgtA8AnlqGm8ypBcNYm9sCNowHU4BhLArj+Z/GM0PHY22YtpnTMkMxMxRjUS5LZl6Mx41J9OMeaZJEURxjEbWXuxMFGRa3OIaf2xumDRAYDzBdFmt8tvEMeT4PsDl9LkOwkeKZiI0UjzVY3HEDRJK4ZxvjAQQ2C1juAL+bB3LKjYkimFVMG7aCcUuSYBbIRXeOxjESnRhe9/xgqySKksRtAZtbQRRhFliNuAVTABowSxSZc/DJeRQczqlg+Ftk8QtQSwMEFAAAAAgACXX7XJeKuxzAAAAAEwIAAAsAAABfcmVscy8ucmVsc52SuW7DMAxAf8XQnjAH0CGIM2XxFgT5AVaiD9gSBYpFnb+v2qVxkAsZeT08EtweaUDtOKS2i6kY/RBSaVrVuAFItiWPac6RQq7ULB41h9JARNtjQ7BaLD5ALhlmt71kFqdzpFeIXNedpT3bL09Bb4CvOkxxQmlISzMO8M3SfzL38ww1ReVKI5VbGnjT5f524EnRoSJYFppFydOiHaV/Hcf2kNPpr2MitHpb6PlxaFQKjtxjJYxxYrT+NYLJD+x+AFBLAwQUAAAACAAJdftcj1kCQK4BAABPBAAADwAAAHhsL3dvcmtib29rLnhtbLVTXWvbMBT9K54w9K123K1jIQ6MZR+Fbg1t6GuQpev4UknXSErT9dfv2p43p2NlDPok3Q/OOfdcaXEgf1cR3SUP1rhQiibGdp5lQTVgZTilFhxXavJWRg79LgutB6lDAxCtyYo8P8+sRCeWixFr7bNpQBFURHKc7BK3CIfwu96FyT0GrNBg/F6K/m5AJBYdWnwEXYpcJKGhwxfy+EguSnOjPBlTitlQuAUfUf2RvulEbmQV+kyU1bVkIaU4zxmwRh9i39HjS9Z4D9w8RPtIn9BE8CsZ4bOnfYtu18HwFNlkjN6H8RxMnPt/sZHqGhWsSO0tuDj46MF0Al1osA0icdJCKa7Z7eQrdBMxxYUepossa+KVnyMX/IXuBb6cmA2oxhEocmRRbTdgW+6aiiueEVe8rLi19HzyzraXsAOnJ6rOnlF11u90XKSGGh3ob4x0HP0k2T4YZ0+3vx6HrGRgNEPd85tsqEGtwXXX5clfXDt5lb5PZ/P0Y1rkxSKbcP03cXFM/NSRkXGVFm+fEGbHgzOoWvukO/qBitdvZu/40+yN+cC5K3dJUo//YfzLyx9QSwMEFAAAAAgACXX7XLts6uy6AAAAGgMAABoAAAB4bC9fcmVscy93b3JrYm9vay54bWwucmVsc8WTOQ6DMBBFr4J8AIYlSREBVRraiAtYMCxiseWZKHD7ECjAUoo0iMr6Y/n9V4yjJ3aSGzVQ3Whyxr4bKBY1s74DUF5jL8lVGof5plSmlzxHU4GWeSsrhMDzbmD2DJFEe6aTTRr/IaqybHJ8qPzV48A/wPBWpqUakYWTSVMhxwLGbhsTLIfvzmThpEUsTFr4As4WCiyh4Hyh0BIKDxQinjqkzWbNVv3lwHqe3+LWvsR1aC/J9esA1ldIPlBLAwQUAAAACAAJdftcpvxKWyMBAADfBAAAEwAAAFtDb250ZW50X1R5cGVzXS54bWzNlM9OwzAMxl+l6nVqMobEAa27AFfYgRcIjbtGzT/F3ujeHrfdJoFGxTQkuDRqbH8/x5+S5es+Amadsx7LvCGK91Ji1YBTKEIEz5E6JKeIf9NGRlW1agNyMZ/fySp4Ak8F9Rr5avkItdpayp463kYTfJknsJhnD2NizypzFaM1lSKOy53XXyjFgSC4csjBxkSccUIuzxL6yPeAQ93LDlIyGrK1SvSsHGfJzkqkvQUU0xJnegx1bSrQodo6LhEYEyiNDQA5K0bR2TSZeMIwfm+u5g8yU0DOXKcQkR1LcDnuaElfXUQWgkRm+ognIktffT7o3dagf8jm8b6H1A5+oByW62f82eOT/oV9LP5JH7d/2MdbCO1vX7l+FU4Zf+TL4V1bfQBQSwECFAMUAAAACAAJdftcRsdNSJUAAADNAAAAEAAAAAAAAAAAAAAAgAEAAAAAZG9jUHJvcHMvYXBwLnhtbFBLAQIUAxQAAAAIAAl1+1wk+zMd7wAAACsCAAARAAAAAAAAAAAAAACAAcMAAABkb2NQcm9wcy9jb3JlLnhtbFBLAQIUAxQAAAAIAAl1+1yZXJwjEAYAAJwnAAATAAAAAAAAAAAAAACAAeEBAAB4bC90aGVtZS90aGVtZTEueG1sUEsBAhQDFAAAAAgACXX7XOg9H21jBAAAEAwAABgAAAAAAAAAAAAAAICBIggAAHhsL3dvcmtzaGVldHMvc2hlZXQxLnhtbFBLAQIUAxQAAAAIAAl1+1xR/7DAXRUAAK/nAAAYAAAAAAAAAAAAAACAgbsMAAB4bC93b3Jrc2hlZXRzL3NoZWV0Mi54bWxQSwECFAMUAAAACAAJdftcUt6roxkJAADmJgAAGAAAAAAAAAAAAAAAgIFOIgAAeGwvd29ya3NoZWV0cy9zaGVldDMueG1sUEsBAhQDFAAAAAgACXX7XBA+UsMsAwAAIxEAAA0AAAAAAAAAAAAAAIABnSsAAHhsL3N0eWxlcy54bWxQSwECFAMUAAAACAAJdftcl4q7HMAAAAATAgAACwAAAAAAAAAAAAAAgAH0LgAAX3JlbHMvLnJlbHNQSwECFAMUAAAACAAJdftcj1kCQK4BAABPBAAADwAAAAAAAAAAAAAAgAHdLwAAeGwvd29ya2Jvb2sueG1sUEsBAhQDFAAAAAgACXX7XLts6uy6AAAAGgMAABoAAAAAAAAAAAAAAIABuDEAAHhsL19yZWxzL3dvcmtib29rLnhtbC5yZWxzUEsBAhQDFAAAAAgACXX7XKb8SlsjAQAA3wQAABMAAAAAAAAAAAAAAIABqjIAAFtDb250ZW50X1R5cGVzXS54bWxQSwUGAAAAAAsACwDKAgAA/jMAAAAA"
_PREP_TEMPLATE_B64 = "UEsDBBQAAAAIAPl1+1xGx01IlQAAAM0AAAAQAAAAZG9jUHJvcHMvYXBwLnhtbE3PTQvCMAwG4L9SdreZih6kDkQ9ip68zy51hbYpbYT67+0EP255ecgboi6JIia2mEXxLuRtMzLHDUDWI/o+y8qhiqHke64x3YGMsRoPpB8eA8OibdeAhTEMOMzit7Dp1C5GZ3XPlkJ3sjpRJsPiWDQ6sScfq9wcChDneiU+ixNLOZcrBf+LU8sVU57mym/8ZAW/B7oXUEsDBBQAAAAIAPl1+1zOR1lX7wAAACsCAAARAAAAZG9jUHJvcHMvY29yZS54bWzNksFOwzAMhl8F5d667cYqRV0uIE4gITEJxC1yvC2iaaLEqN3b05atE4IH4Bj7z+fPkhsMEn2k5+gDRbaUbgbXdkli2Iojc5AACY/kdMrHRDc29z46zeMzHiBo/NAHgqooNuCItdGsYQJmYSEK1RiUGEmzj2e8wQUfPmM7wwwCteSo4wRlXoJQ08RwGtoGroAJxhRd+i6QWYhz9U/s3AFxTg7JLqm+7/N+NefGHUp4e3p8mdfNbJdYd0jjr2QlnwJtxWXy6+rufvcgVFVUm6yos6relWu5ruVt8T65/vC7Cjtv7N7+Y+OLoGrg112oL1BLAwQUAAAACAD5dftcmVycIxAGAACcJwAAEwAAAHhsL3RoZW1lL3RoZW1lMS54bWztWltz2jgUfu+v0Hhn9m0LxjaBtrQTc2l227SZhO1OH4URWI1seWSRhH+/RzYQy5YN7ZJNups8BCzp+85FR+foOHnz7i5i6IaIlPJ4YNkv29a7ty/e4FcyJBFBMBmnr/DACqVMXrVaaQDDOH3JExLD3IKLCEt4FMvWXOBbGi8j1uq0291WhGlsoRhHZGB9XixoQNBUUVpvXyC05R8z+BXLVI1lowETV0EmuYi08vlsxfza3j5lz+k6HTKBbjAbWCB/zm+n5E5aiOFUwsTAamc/VmvH0dJIgILJfZQFukn2o9MVCDINOzqdWM52fPbE7Z+Mytp0NG0a4OPxeDi2y9KLcBwE4FG7nsKd9Gy/pEEJtKNp0GTY9tqukaaqjVNP0/d93+ubaJwKjVtP02t33dOOicat0HgNvvFPh8Ouicar0HTraSYn/a5rpOkWaEJG4+t6EhW15UDTIABYcHbWzNIDll4p+nWUGtkdu91BXPBY7jmJEf7GxQTWadIZljRGcp2QBQ4AN8TRTFB8r0G2iuDCktJckNbPKbVQGgiayIH1R4Ihxdyv/fWXu8mkM3qdfTrOa5R/aasBp+27m8+T/HPo5J+nk9dNQs5wvCwJ8fsjW2GHJ247E3I6HGdCfM/29pGlJTLP7/kK6048Zx9WlrBdz8/knoxyI7vd9lh99k9HbiPXqcCzIteURiRFn8gtuuQROLVJDTITPwidhphqUBwCpAkxlqGG+LTGrBHgE323vgjI342I96tvmj1XoVhJ2oT4EEYa4pxz5nPRbPsHpUbR9lW83KOXWBUBlxjfNKo1LMXWeJXA8a2cPB0TEs2UCwZBhpckJhKpOX5NSBP+K6Xa/pzTQPCULyT6SpGPabMjp3QmzegzGsFGrxt1h2jSPHr+BfmcNQockRsdAmcbs0YhhGm78B6vJI6arcIRK0I+Yhk2GnK1FoG2camEYFoSxtF4TtK0EfxZrDWTPmDI7M2Rdc7WkQ4Rkl43Qj5izouQEb8ehjhKmu2icVgE/Z5ew0nB6ILLZv24fobVM2wsjvdH1BdK5A8mpz/pMjQHo5pZCb2EVmqfqoc0PqgeMgoF8bkePuV6eAo3lsa8UK6CewH/0do3wqv4gsA5fy59z6XvufQ9odK3NyN9Z8HTi1veRm5bxPuuMdrXNC4oY1dyzcjHVK+TKdg5n8Ds/Wg+nvHt+tkkhK+aWS0jFpBLgbNBJLj8i8rwKsQJ6GRbJQnLVNNlN4oSnkIbbulT9UqV1+WvuSi4PFvk6a+hdD4sz/k8X+e0zQszQ7dyS+q2lL61JjhK9LHMcE4eyww7ZzySHbZ3oB01+/ZdduQjpTBTl0O4GkK+A226ndw6OJ6YkbkK01KQb8P56cV4GuI52QS5fZhXbefY0dH758FRsKPvPJYdx4jyoiHuoYaYz8NDh3l7X5hnlcZQNBRtbKwkLEa3YLjX8SwU4GRgLaAHg69RAvJSVWAxW8YDK5CifEyMRehw55dcX+PRkuPbpmW1bq8pdxltIlI5wmmYE2eryt5lscFVHc9VW/Kwvmo9tBVOz/5ZrcifDBFOFgsSSGOUF6ZKovMZU77nK0nEVTi/RTO2EpcYvOPmx3FOU7gSdrYPAjK5uzmpemUxZ6by3y0MCSxbiFkS4k1d7dXnm5yueiJ2+pd3wWDy/XDJRw/lO+df9F1Drn723eP6bpM7SEycecURAXRFAiOVHAYWFzLkUO6SkAYTAc2UyUTwAoJkphyAmPoLvfIMuSkVzq0+OX9FLIOGTl7SJRIUirAMBSEXcuPv75Nqd4zX+iyBbYRUMmTVF8pDicE9M3JD2FQl867aJguF2+JUzbsaviZgS8N6bp0tJ//bXtQ9tBc9RvOjmeAes4dzm3q4wkWs/1jWHvky3zlw2zreA17mEyxDpH7BfYqKgBGrYr66r0/5JZw7tHvxgSCb/NbbpPbd4Ax81KtapWQrET9LB3wfkgZjjFv0NF+PFGKtprGtxtoxDHmAWPMMoWY434dFmhoz1YusOY0Kb0HVQOU/29QNaPYNNByRBV4xmbY2o+ROCjzc/u8NsMLEjuHti78BUEsDBBQAAAAIAPl1+1y9BtDF3gMAAJ4KAAAYAAAAeGwvd29ya3NoZWV0cy9zaGVldDEueG1snVbbbts4EP2VgR7qLlBblu1cmtoG2iRtAjQbo0nTR4OWxhIRitSSlB3/fYfUBQGykYI8WOZlOJczh8OZ75V+NBmihadcSLMIMmuLszA0cYY5MyNVoKSdrdI5szTVaWgKjSzxh3IRTsbj4zBnXAbLuV9b6eVclVZwiSsNpsxzpg/fUKj9IoiCZuEXTzPrFsLlvGAp3qH9Xaw0zcJWS8JzlIYrCRq3i+BrdPY1mrkDXuKB4948G4PJ1P6H5slPskyBjANwwW2UenTb14lbInkUGFunlNHfDs9RCKebPPuvNhO0XriDz8eNve8eDgpvwwyeK/GHJzZbBKcBJLhlpbC/1P4K6xCPnL5YCeO/sK9ko4gcjEtjVV6fJhdyLqt/9lRjE1YHvdULZtlyrtUetJfy2k8bLa09CjJ2Ej4mL0irXLqE3FlNu5wU2uW5KqXVB1hpLOA7FwhDuMe8EMziPLRk2smFMf3IZGt30m934u1OvF1Hixc6ppWO6ex1HdNnOl71fShwhwKMirkaYqykynkMTCaQIRM2Ay6L0kJCuAExGGyGcCvv7m8fLiEmXnCZDlGiTg+QqwTFGRSqKAkAoscnKPWGSW7qmaeN5lse1wu1iU2pE6Sps9q6UDDNcrSozagDylk/lLM+KI/6oTzqhPI+Q4NgCozb2AwwjQRQhXGzRfTYMVGiIRiZpXvMLSjZyhGvhQCp6h1Gowz1CH6Tdod7rgytxnGpiV8EbjG0akiZwSo9bMe4YBtiYZOoRu8GKU1VfgQmXXge9+N53IfnSb+OE69j+gqeV6THKih92Ny0NajD79N+m6ddOXSl+8wULMZFQLXZoN5hsASIRu5eC7oFHtAHlz2CVZS5pMzBwN38dXPlB5RAovOLrLvsdIH+ud/5z+9xflJRZ7BqrtL6J6YokwEw4ic84sEzZe+4iCzOwAtCjkyaT8CtqXhlDwVWl7OU3HYF4upxb0UdvyeU6Qgunwql7QvIKZQPwn65vrudfkjtl3XhtrdUi0ex2fn4HI8of3XV6vT/LS9C1Mf/6A31PZp03oB/lUXT5ej0DSY66/9rQA/hQskB8SGhCuQo7yiwdhSoad+J36y/lEaz97l1Q8lfh/TdrMHVP+MZeX573ZRU6l1KkZCXORVArXLv/QZdyWwLo/TFmQkgWmhMq3H1Cg2rV6ii/EccpSO4vrq5DH98u/inLagJN0jtClBz5uwYIPcfMXHViqwbzBR5wLimR1CI0tn6X7jCZ72I69xumE45vRkCtwTKeHRC742ugKsmVhW+C9koS6BWnQs1kKidAO1vFRGmnriOp21Jl38BUEsDBBQAAAAIAPl1+1xZMe3iYgUAAE4hAAAYAAAAeGwvd29ya3NoZWV0cy9zaGVldDIueG1spZrtbuI4FIZvJcoFDCROaTsCpCmdUavtbFGZ6f6MDBiwJomzjmmGufp1DI1R1+ecSv3Tkg+/fuLW50kcxq3Sv5qdECb6XRZVM4l3xtSfB4NmtRMlbz6pWlT2yEbpkhu7qbeDptaCr12jshikw+FoUHJZxdOx2zfX07Ham0JWYq6jZl+WXB9uRKHaSZzErzue5HZnuh2D6bjmW7EQ5mc913Zr0KesZSmqRqoq0mIzib8kn2fZVdfAnfEsRducfY66S1kq9avbuF9P4mHcRVciOizqQrrOIqPqB7ExM1EUNjCNI74y8kXM7WmTeKmMUWV33GIabuyujVZ/ROX6FIWw51qY+n8nH0NOod01/nsCjvvr6aDOP7+Sf3MDawdqyRsxU8U/cm12k/gqjtZiw/eFeVLtnTgN1kWXt1JF435G7fFcNoyj1b6xNKfGlqCU1fE3/30a5LMGyQhokJ4apG8bQD2wUwPmLvRI5i7rlhs+HWvVRro726Z1H9zYTOLM/h0msay6f5GF0faotO3MdM41L8cDY6O6HYPVqdkN3uyZF3sRaDbDm60tYm4O9ZumAwvdk6c9eeqyLoCsmdpXRh/yipchkptj65Fr3U0Uz4jnNkbLaosAsh6QvQtwpdZBQAYC4rkkYNYDZmjQwnBt8oPgOoSXgXh4qqwMwnbRs12gKV+rNUh2AZLhmTjZqCcboSlzVe8L3lWlvEEHcARi4h3gmJc95uV7MQUylpcgJB6PQ171kFdoyk+95KdhDMFdgXB47KZQHMO77vGu34Fnhy8Edw3C4aEUXDL0pXuIzxFrx1xbZwbr9xDkI2JJwDO3JGiS3mte5ALFTGBMPJzE9CJJ8Iq/P/6ZUUzYJkQ4iel1kuB1/7uyxWb2OL8NAsI2IWJJQK+TBK/8DvD+LswH64RIJfm8UhK8/ju+h1kQD3YKEUriea8keN13eF8enu6DgLBNiFgS0BslwWu+A1z8eHr862sQEXYJEUwiep8keOW3iEt4jsA+IWJJQG+UBK/+DhCaI7BSiFSKL/VOSfHi7/jCcySFjUKEknjeKCle9B0eNEdS2CVELAl49lCCl3sHCM+RFHkq+ZhHUu+RFC/4T87Kd3eN/BN+cIJVQiSTjF4lKV70j3deCCOsEyKZZPQ6SQmdCF40eS10vuaHICQsFSKahPRSSamHiY3meetWM4KMsFeIZJLReyXFy//fPx4wQlgrRC5J6LWS4vW/VjVGCHuFyCUJvVdSwgCVXgbZYKdQiQQb805hePmfPd6DNZvBUiFSST4vFYZX/44PumdgsFOIVJLPO4URK12WD7hlYLBPiFAS72yZi1jnsnjhOwaGLHF9zCTMm4Th9b6jg33MYI8QuSSh9wjDi/1MNSZSm2jF9VJVkShl0y3AN0FeWClELySvVwrDC/+tbFbd0ib4sMxgpxDRJKR3CsNr/52Vs9nlTS2LQr0IbUXdrXQLI4LLYAyWDNERiewlw4hnF1nJcl/mLd+GhxXWDJFMMnrNMFwKz4uHIBpsGSKQQsu8ZTLcB22+shMpOGsyWDJEKInnJZPhOmhzUb1IrapSvF06PUHCpiGiSUhvmgyXQpvv3LwJ8sGqIVJJPq+aDNdCezahg4ywcIhkkvHspQouhjY3MvxGKkNeqXxMNpmXTYZrwI4gL16ACpPBdiFiSUBvlwxXwHZd5/UqSAdrhcgk6bxWMrzab22NDrLB/iASSTbvjwyv8ttK2pHL67oOEsL2IHJBwsHZa2e+N+qbLKxe335xoPumwXeu7cg1USE2tovhp0vblz6+YD9uGFW7N9zHN/zuo61Ea6G7E+zxjVLmdaN77d1/hWL6H1BLAwQUAAAACAD5dftcFChWKMAJAABgOgAAGAAAAHhsL3dvcmtzaGVldHMvc2hlZXQzLnhtbK2bbW/bthqG/wrhAxxsQBNbkvO2kwRI7TYJljVGsnQfDVqiba6SqJFU3OzXn4eUbXkZ9dBh82W1Hd0SeY0ir0cv5yshv6klY5p8L/JSXfSWWle/9PsqXbKCqkNRsRL+MheyoBq+ykVfVZLRzIaKvB8PBsf9gvKyd3luf5vIy3NR65yXbCKJqouCypePLBeri17U2/zwwBdLbX7oX55XdMEemX6qJhK+9bd7yXjBSsVFSSSbX/Suol/Gw1MTsFt85Wyldj4T05WZEN/Ml9vsojfomV2XjLw8Vjm3ByNaVHdsrkcsz2GHcY/QVPNnNoHNLnozobUozN+hmZpq+Gkuxd+stMdkOYNtoTHVvzZudrLeqenjX+sG97b9MY3a/bxp+WcLFkDNqGIjkf/BM7286J32SMbmtM71g1jdsDWsI7O/VOTK/pesmm1j2DitFbRmHYYWFLxs/qXf15B3AtGgIxCvA/GrQNwVSNaB5FXgrCswXAeGlkzTFcthTDW9PJdiRaTZGvZmPliYNgXd56UZU49awl855PTlhEpanPc17Mr80E/XsY94LINjTfVLxRzRER59Krl2pMZ4asxUKnllRs8/w33o8LbX8bbXsd3bScfeRqIutXyZlrRw9eBjkz7qSCsteblw9RzP/fc/p3EU/c/V+SZ42hH8XOc5MW0lYk70kpG0af8hgiLZokj2QpGKzIkiCUSB5xAUCYri9vE+2XSfmDaTn9jh4pD8+unLB3J1ff+BPF1f/YyBGW7BDFEwj5pKPX1hVLqwDNHu8dI1xEd4CGEyxIcHl0oT09LN+CgATA6bLUjFJBcZxuNoy+MI5fGpzDppHIXQwEMIjSMPjZLmwTSOtzSOURoTUdU5NdPRVKED5TgEDR6aMFHlrpN1fIyS+WLbC3CqbeMJ1RaR7YOFhsE52cI52RcOQ0bNSQgaPNSN5gRFM5HiT3ATlsE02wkJ+uJFdLpFdIoiepIzuh46LjSnaC/nuaBOOHjswfSGDA4iF59TlM/jksrt6rNDJs2pUnzOARxVpDadetuQOtvyOtuDF/wfcNE6C6OFx1BaZ3uOJvV2bvuMsmjQ6t0An7VBuacSRNzpeIMwcJ4cSm6d9c5RtlaQgChtiJk+oEh2jDdCkcha0nzKUDBRIBg8h4OJUDAPptVvp9IacYQrcd2cYCgVXG+7qeA5nApuxnZaeDuVVo4j3I5/E7C0j+4nYycP3HG7eeC5MaN6qYyrkGgw+NZ/cS2e4wj3ZNNwCiX7i4UBcwvwmdWaznIGFTwxXSJQmRdkKWrFliKHiYhLmKLyvDYMUXytQke4Q1t8tzduergNd9PDc3vSw43aQ4+rdElZwVOyZGaRy7hiVOFDrtXsCPdsy+xu5ESGK3M3Mjy3JzJcuz3I8hr0O6VlyvCFrfXvCBdwi+nq7uHWCQoX6G5QeG5PULiFe0DRtIZfc7GCg0imKg5bCah1eTlvLp6h+FpDj3BFt/gef3+4//WTEyCu2d0A8dyeAHFX9wBUWopv+LnYSnqEWzocatY9/QdKuic3gpnEjwjXddNunhlEP/E8L5lScO4p9nNDbC6kXQBQRq2YR7iZW0Zdc3ygmXty+yHCHb1FtGUSMK3HrYjHuIhbTO5pPQ7UcE9uL0oxruMOSnvO5HHr4zHu45ZM10weB5q4J7cfG9zIHWx+aPKOd65e465uiXVP3nGgp3ty+zHDfd3BzD9fx62ux7iu2yJpenOj+N/u6/qBxu7JNVed+jc3TiK4qF89M0kXjNiqdMfGTRdQKK2Ex7iEN5dOECiBIu7J4VBw/95AaS6MvAFKa9mxx7IZzdUUBvM0oy9OKoGu7cnZ4/bhuEqUffehxzFu2hs4ZV3MmL3IXZidklTAeZTZE7TZv/0Ix0CJtcId48INc5ik05W9keoEFujcnhx20wwX7T9sU8mCP7PSiCH7zpU2dwFsT2DaqVNdS0YqKb7zwsxJvGyuwL26nJCCbc4k9c7frXzHuHx/+f0OAxno3p4cBhIX7tcgS/NV8wJWOvNJ/Si3VsdjXMcrUWHcAn3ck8O44Rb+mtvO9d/MPJXx4yOulfQYl/R5KWdOZoGC7smhl/RiXMw/S9o8JAITWynKA8lKtrLF3YyLgir1gcBykNkzGs7nwngESLsyT7mo5prWekOypPKZ2XMevVPeOnyCO/zo/rZTVJNAiffknh7HfVO6Oe+U4+4+EkobiJsK0KwGpvW2FESBtOqe4OpugHSVx0mguXtyKBBc2F1A7IVQL5DWzBPPgyUApKMWTgKt3JNDeeAy7uIBjffj2Hm2xPNwCeBw17xJoIl7cigNXMRdNHZqXT+VVsgTXMgNle7qLQnUcU8OJYPbuItMU7X5obRCnuBCvjlISuUMJv/tjO5EFOjmnpxBpEVZMpgVYicnXMwflzSDbleSp2ynK5vVqhAl01ztrFYouVbME1zMx1zZB6I679slgWbuyaGLfIK7+abN60u4VZWbW+FgQ4DooJJMsVIfPNO8ZqQPhQ0MDVCidG1POLfWwhPcwm+gatLLqapgXAuoqaAcNA9DMs2cT6skgVruyWFPwnmeWdm0lijjizBTLW2PDmZgTHOuSdsza+3gUjA+K1sCtSV1wUwViSNtBT3xXC/nJS/qYrqC+tTJMFDRPTlz4sLZpZdOirikbx84KJq2E9P2D/8+aW29o+gz0MNhtU6e4E7+9fHOyShQyT05wyjnc/f8jxv5V3sewoRG7TPdpoqGAUfM3hyg6LPgGfxWbG7ToA9jtu49xN17NTXTgHM5GAaqtyeHPY+Jm3dT+JnzbFv72cZvyr2izjU/SCWHk5dTolm6LEUuFi8kXQpYPFBirZwPcTlfTVn5zKUoC+Z8lO7jMNDQPTmMGy7oDm47XYAhx4sKisN3wdgq/RBX+tW0mVadBAOd3pPDCOJK7yDYtP490bX6P8T1f7WzuDrxBRYBnhyGD68BHPi2PSDrZfV9zuKdp9DxYmE1NSuPk15goeDJYfTwOsFBb3fVfBdubT0xxOsJGHs0f+7wkGFgAeHJYejw0sE18Jrm90GGeVbD5Nfo8HtAbEuLIV5aLLJqWqVOgoE1hSfXVKkV19QJES8qrscTW5s2O0AJtEXCEC8SFmCFzv4HVgOeHFpTDfF64BqaCp7B5nOeclgvbclepqIwg4b9VftlrPX8Ie75i5LDuJhWVeVkE2j5nhyMDTKZTLDxgav+9ZfbnfHxwewMKqI/a6WZ+52Q/s57dbTW4jPPtb2z/49XKc27l79RCSNFkZzN4dCDwxPohGxeOWy+aFHZV/iadx7tR1iYMybNBvD3uRB688W817d9qfTy/1BLAwQUAAAACAD5dftc9uBcnTEDAAAjEQAADQAAAHhsL3N0eWxlcy54bWzdWO1umzAUfRXEA4wEWhqmJFIbLdKkbarU/thfJ5jEksHMmI7s6edrEyCNb5as7VQNFGH7+txzfH39oUwrteP0YUup8pqcF9XM3ypVfgyCar2lOak+iJIW2pIJmROlq3ITVKWkJK0AlPMgHI3iICes8OfTos6Xuaq8tagLNfNHfjCfZqLoW25826C7kpx6T4TP/AXhbCWZ6Utyxne2OYSGteBCekpLoTN/DC3VL2se2xqobP3krBASGgPL8JznVjLCwb5qPfQEcrPSakfj5dWnm8kBS3yOw0NNr0kfvsjh0jyXK+x6jy6ldyLNp9IeGOddJlz5tmE+LYlSVBZLXTEY03hk8try467UqbCRZDcOr/2zAZXgLAXKzcId8mAAfaFTHfNwsUCdmo8Ox0rIlMouIKG/b5pPOc2Uhku22cJXiRLiLJQSuS6kjGxEQUy09ogh0jOLeuarrVmUBylxt4TXaIOuLceZCNPXyDkToHvudZ+JsJ0HA2sLOl5ryvkDOPmedUEba1dN5tl953MKW44H2bYv6ki3RevGVoBo6M36HrhN/sqtV7Inoe5qPYLC1H/UQtF7STPWmHqTdfyY9zHunZQl391ytilyasd+NuF8SvY474lKxdawSPXs+N5PScpH2qh2AQdNhosL37O46D2Lu+rFhUNx4zcQt9ZVKv8o6RqJ1wX8J/J4sEqi1/cev6n2E5H5d5kUtFvSYN872PW6Vg+O5Jn/DS5ovKfzVjXjihVtbcvSlBZHm592r8hK3wAP/Ov+Kc1IzdVjZ5z5ffkrTVmdJ12vewhB26svf4HTwl6gzG6vuViR0oami7aqt/+Dg9M+AHhu6W8yxxYMY21uC9gwHkwBhrEojOd/Gs8EHY+1YdomTssExUxQjEW5LAvzYjxuTKIf90iTJIriGIuovdwdKVhgcYtj+Lm9YdoAgfEA02Wxxmcbz5DTeYDN6akMwUaKZyI2UjzWYHHHDRBJ4p5tjAcQ2CxguQP8bh7IKTcmimBWMW3YCsYtSYJZIBfdORrHSHRieN3zg62SKEoStwVsbgVRhFlgNeIWTAFowCxRZM7BZ+dRsD+ngv5vkflvUEsDBBQAAAAIAPl1+1yXirscwAAAABMCAAALAAAAX3JlbHMvLnJlbHOdkrluwzAMQH/F0J4wB9AhiDNl8RYE+QFWog/YEgWKRZ2/r9qlcZALGXk9PBLcHmlA7TiktoupGP0QUmla1bgBSLYlj2nOkUKu1CweNYfSQETbY0OwWiw+QC4ZZre9ZBanc6RXiFzXnaU92y9PQW+ArzpMcUJpSEszDvDN0n8y9/MMNUXlSiOVWxp40+X+duBJ0aEiWBaaRcnToh2lfx3H9pDT6a9jIrR6W+j5cWhUCo7cYyWMcWK0/jWCyQ/sfgBQSwMEFAAAAAgA+XX7XFZ6iCKkAQAAOgQAAA8AAAB4bC93b3JrYm9vay54bWy1U+9r2zAQ/Vc8Eei3KnG7soU4MBq2FbottKVfg2yd46P6YU5K0/Wv38meN6eFUgr9JN2deO/du9Ni7+mu9P4ue7DGhUI0MbZzKUPVgFXh2LfguFJ7sipySFsZWgKlQwMQrZH5dHomrUInlosBa01yHPgIVUTvOJkStwj78L+ewuweA5ZoMP4uRHc3IDKLDi0+gi7EVGSh8fvvnvDRu6jMdUXemELM+sItUMTqWfo6ibxRZegyUZVXioUU4mzKgDVSiN2LDl+xxnvgx320i/4rmgi0UhG+kd+16LYJhruQozY6H4azN3FOr7HR1zVWsPLVzoKLvY8EJgl0ocE2iMwpC4W4YrezH5A6YooL3XcXWdbIK5ojF+hCdwLfT8yaoN3cgG25NpaUvyApf2dJivjkSW0uYQtOj1SdvKDqpJvkMD4NNTrQPxnpMPpLsnkwzh5v/q2EKlVgNOPT0o3m0qDW4NJ1eXTg1dGHyZfJbD45n5x+WsgRwZvZ8idsT2wYCFfPCeVhtwxarSlLR9dFfvpx9pn/x86Yc879cpde6WH1h2+7/ANQSwMEFAAAAAgA+XX7XLts6uy6AAAAGgMAABoAAAB4bC9fcmVscy93b3JrYm9vay54bWwucmVsc8WTOQ6DMBBFr4J8AIYlSREBVRraiAtYMCxiseWZKHD7ECjAUoo0iMr6Y/n9V4yjJ3aSGzVQ3Whyxr4bKBY1s74DUF5jL8lVGof5plSmlzxHU4GWeSsrhMDzbmD2DJFEe6aTTRr/IaqybHJ8qPzV48A/wPBWpqUakYWTSVMhxwLGbhsTLIfvzmThpEUsTFr4As4WCiyh4Hyh0BIKDxQinjqkzWbNVv3lwHqe3+LWvsR1aC/J9esA1ldIPlBLAwQUAAAACAD5dftcpvxKWyMBAADfBAAAEwAAAFtDb250ZW50X1R5cGVzXS54bWzNlM9OwzAMxl+l6nVqMobEAa27AFfYgRcIjbtGzT/F3ujeHrfdJoFGxTQkuDRqbH8/x5+S5es+Amadsx7LvCGK91Ji1YBTKEIEz5E6JKeIf9NGRlW1agNyMZ/fySp4Ak8F9Rr5avkItdpayp463kYTfJknsJhnD2NizypzFaM1lSKOy53XXyjFgSC4csjBxkSccUIuzxL6yPeAQ93LDlIyGrK1SvSsHGfJzkqkvQUU0xJnegx1bSrQodo6LhEYEyiNDQA5K0bR2TSZeMIwfm+u5g8yU0DOXKcQkR1LcDnuaElfXUQWgkRm+ognIktffT7o3dagf8jm8b6H1A5+oByW62f82eOT/oV9LP5JH7d/2MdbCO1vX7l+FU4Zf+TL4V1bfQBQSwECFAMUAAAACAD5dftcRsdNSJUAAADNAAAAEAAAAAAAAAAAAAAAgAEAAAAAZG9jUHJvcHMvYXBwLnhtbFBLAQIUAxQAAAAIAPl1+1zOR1lX7wAAACsCAAARAAAAAAAAAAAAAACAAcMAAABkb2NQcm9wcy9jb3JlLnhtbFBLAQIUAxQAAAAIAPl1+1yZXJwjEAYAAJwnAAATAAAAAAAAAAAAAACAAeEBAAB4bC90aGVtZS90aGVtZTEueG1sUEsBAhQDFAAAAAgA+XX7XL0G0MXeAwAAngoAABgAAAAAAAAAAAAAAICBIggAAHhsL3dvcmtzaGVldHMvc2hlZXQxLnhtbFBLAQIUAxQAAAAIAPl1+1xZMe3iYgUAAE4hAAAYAAAAAAAAAAAAAACAgTYMAAB4bC93b3Jrc2hlZXRzL3NoZWV0Mi54bWxQSwECFAMUAAAACAD5dftcFChWKMAJAABgOgAAGAAAAAAAAAAAAAAAgIHOEQAAeGwvd29ya3NoZWV0cy9zaGVldDMueG1sUEsBAhQDFAAAAAgA+XX7XPbgXJ0xAwAAIxEAAA0AAAAAAAAAAAAAAIABxBsAAHhsL3N0eWxlcy54bWxQSwECFAMUAAAACAD5dftcl4q7HMAAAAATAgAACwAAAAAAAAAAAAAAgAEgHwAAX3JlbHMvLnJlbHNQSwECFAMUAAAACAD5dftcVnqIIqQBAAA6BAAADwAAAAAAAAAAAAAAgAEJIAAAeGwvd29ya2Jvb2sueG1sUEsBAhQDFAAAAAgA+XX7XLts6uy6AAAAGgMAABoAAAAAAAAAAAAAAIAB2iEAAHhsL19yZWxzL3dvcmtib29rLnhtbC5yZWxzUEsBAhQDFAAAAAgA+XX7XKb8SlsjAQAA3wQAABMAAAAAAAAAAAAAAIABzCIAAFtDb250ZW50X1R5cGVzXS54bWxQSwUGAAAAAAsACwDKAgAAICQAAAAA"

def get_specification_templates(country):
    import base64
    out = _mkdir(f'Data/{country}/Specifications')
    files = {
        f'{out}/{country}_technoeconomic_TEMPLATE.xlsx': _TECHNOECON_TEMPLATE_B64,
        f'{out}/{country}_prep_file_TEMPLATE.xlsx':      _PREP_TEMPLATE_B64,
    }
    for path, b64 in files.items():
        with open(path, 'wb') as f:
            f.write(base64.b64decode(b64))
        print(f'[Templates] Saved -> {path}')
    print('[Templates] These are BLANK TEMPLATES (Read Me + Parameter Legend tabs included) -')
    print('            fill in the Value column with data for this country before use in OnSTOVE.')

print('Downloader functions loaded')

Downloader functions loaded


In [3]:
# @title Step 3: Select country and datasets {"vertical-output": true, "display-mode": "form"}
# @markdown Select a country and tick the datasets to download, then click **Get Data**.

from google.colab.output import eval_js
eval_js('google.colab.output.setIframeHeight(0, true, {maxHeight: 3000})')

display(HTML('<link rel="stylesheet" href="https://stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css">'))

# Inline CSS injected via HTML widget - avoids Python parse issues with CSS colons
CSS_PARTS = [
    '<style>',
    '.sdk-card {',
    '  background-color: var(--colab-primary-surface-color);',
    '  border: 1px solid var(--colab-border-color);',
    '  border-radius: 12px; padding: 18px; margin: 10px 0;',
    '  box-shadow: 0 2px 4px rgba(0,0,0,0.08);',
    '}',
    '.sdk-all-box {',
    '  background-color: var(--colab-secondary-surface-color);',
    '  border: 2px dashed var(--colab-border-color);',
    '  border-radius: 10px; padding: 10px 15px; margin-bottom: 20px;',
    '}',
    '.sdk-warn {',
    '  background: #fff3cd; border: 1px solid #ffc107;',
    '  border-radius: 8px; padding: 14px; margin: 10px 0;',
    '  font-family: Segoe UI, sans-serif; font-size: 13px;',
    '}',
    '.p-Collapse-header, .lm-Collapse-header {',
    '  color: #1a73e8 !important; font-size: 18px !important;',
    '  font-weight: 800 !important; border-bottom: 2px solid #d1d5db !important;',
    '}',
    '.borderless * { border: none !important; box-shadow: none !important; }',
    '.widget-checkbox > label > span { font-size: 15px !important; font-weight: 600 !important; }',
    '</style>',
]
display(HTML(''.join(CSS_PARTS)))

COUNTRIES = [
    'Select country',
    'AGO','BDI','BEN','BFA','BWA','CAF','CIV','CMR','COD','COG',
    'DJI','DZA','EGY','ERI','ETH','GAB','GHA','GIN','GMB','GNB',
    'GNQ','KEN','LBR','LBY','LSO','MAR','MDG','MLI','MOZ','MRT',
    'MWI','NAM','NER','NGA','RWA','SDN','SEN','SLE','SOM','SSD',
    'SWZ','TCD','TGO','TUN','TZA','UGA','ZAF','ZMB','ZWE',
    'BRN','IDN','KHM','LAO','MMR','MYS','PHL','SGP','THA','TLS','VNM',
]

country_w = widgets.Dropdown(
    options=COUNTRIES, value='Select country',
    description='Country:', layout=Layout(width='300px'),
    style={'description_width': '80px'}
)

upload_btn = Button(description=' Upload boundary manually', icon='upload',
                     button_style='', layout=Layout(width='230px'))
upload_status = widgets.HTML(value=(
    '<span style="color:#5f6368;font-size:12px;">'
    'Only needed if the automatic GADM fetch keeps failing. Accepts a GADM '
    'level-0 (whole-country) boundary as <code>.gpkg</code>, <code>.geojson</code>/'
    '<code>.json</code>, or a zipped shapefile (<code>.zip</code>). '
    'Get one manually from <a href="https://gadm.org/download_country.html" target="_blank">'
    'gadm.org/download_country.html</a> (choose your country, then GeoPackage/Shapefile/GeoJSON, '
    '"country" / level-0 layer only) if geodata.ucdavis.edu is unreachable from here too.'
    '</span>'
))

def _read_boundary_upload(fname, content_bytes):
    """Parse an uploaded boundary file (gpkg/geojson/json/zip-shapefile) into a GeoDataFrame."""
    import tempfile
    suffix = os.path.splitext(fname)[1].lower()
    with tempfile.TemporaryDirectory() as tdir:
        raw_path = os.path.join(tdir, fname)
        with open(raw_path, 'wb') as f:
            f.write(content_bytes)
        if suffix == '.zip':
            extract_dir = os.path.join(tdir, 'extracted')
            with zipfile.ZipFile(raw_path, 'r') as z:
                z.extractall(extract_dir)
            shp_files = glob.glob(os.path.join(extract_dir, '**', '*.shp'), recursive=True)
            if not shp_files:
                raise Exception('No .shp file found inside the uploaded zip.')
            gdf = gpd.read_file(shp_files[0])
        elif suffix in ('.gpkg', '.geojson', '.json'):
            gdf = gpd.read_file(raw_path)
        else:
            raise Exception(f'Unsupported file type "{suffix}". Use .gpkg, .geojson/.json, or a zipped .shp.')
    if gdf.empty:
        raise Exception('The uploaded file has no features.')
    # Collapse to a single country-outline geometry in case the file has multiple rows/parts
    gdf = gdf.dissolve().reset_index(drop=True)
    return gdf

def on_upload_click(_):
    country = country_w.value
    if country == 'Select country':
        upload_status.value = ('<span style="color:#d93025;">Select a country from the dropdown '
                                'above first, so the upload can be matched to it.</span>')
        return
    with log:
        print(f'--- Manual boundary upload for {country} ---')
        try:
            uploaded = cfiles.upload()
        except Exception as e:
            print(f'Upload cancelled or failed: {e}')
            return
        if not uploaded:
            print('No file was uploaded.')
            return
        fname, content_bytes = next(iter(uploaded.items()))
        try:
            gdf = _read_boundary_upload(fname, content_bytes)
            set_manual_boundary(country, gdf)
            out = _mkdir(f'Data/{country}/Boundaries')
            p = f'{out}/{country}_boundary.gpkg'
            gdf.to_file(p, driver='GPKG')
            print(f'[Boundaries] Manual boundary registered for {country} and saved -> {p}')
            print('This will now be used automatically instead of fetching from GADM for this session.')
            upload_status.value = (f'<span style="color:#188038;">Manual boundary for <b>{country}</b> '
                                    f'loaded from "{fname}" and will be used for this session.</span>')
        except Exception as e:
            print(f'Could not read uploaded boundary: {e}')
            upload_status.value = f'<span style="color:#d93025;">Upload failed: {e}</span>'

upload_btn.on_click(on_upload_click)

temp_upload_btn = Button(description=' Upload Temperature raster manually', icon='upload',
                          button_style='', layout=Layout(width='260px'))
temp_upload_status = widgets.HTML(value=(
    '<span style="color:#5f6368;font-size:12px;">'
    'Only needed if the automatic Global Solar Atlas download keeps failing. Accepts a single '
    '<code>.tif</code>/<code>.tiff</code> raster (e.g. just TEMP.tif), or a <code>.zip</code> '
    'containing the whole GSA GEOTIFF folder. Get one manually from '
    '<a href="https://globalsolaratlas.info/download" target="_blank">globalsolaratlas.info/download</a> '
    '(select country, download the GIS_LTAym_YearlyMonthlyTotals GEOTIFF package).'
    '</span>'
))

def _record_manual_success(country, label):
    """Update RUN_STATUS so a manual upload clears any earlier failure for this
    dataset and shows up correctly in the Instructions.txt file at packaging time."""
    if 'RUN_STATUS' not in globals():
        return
    status = RUN_STATUS.setdefault(country, {'requested': [], 'succeeded': [], 'failed': []})
    status['failed'] = [(lbl, err) for lbl, err in status['failed'] if lbl != label]
    manual_label = f'{label} (manual upload)'
    if manual_label not in status['succeeded']:
        status['succeeded'].append(manual_label)

def on_temp_upload_click(_):
    country = country_w.value
    if country == 'Select country':
        temp_upload_status.value = ('<span style="color:#d93025;">Select a country from the dropdown '
                                     'above first, so the upload can be matched to it.</span>')
        return
    with log:
        print(f'--- Manual Temperature raster upload for {country} ---')
        try:
            uploaded = cfiles.upload()
        except Exception as e:
            print(f'Upload cancelled or failed: {e}')
            return
        if not uploaded:
            print('No file was uploaded.')
            return
        fname, content_bytes = next(iter(uploaded.items()))
        suffix = os.path.splitext(fname)[1].lower()
        out = _mkdir(f'Data/{country}/Temperature')
        try:
            if suffix in ('.tif', '.tiff'):
                dest = f'{out}/{fname}'
                with open(dest, 'wb') as f:
                    f.write(content_bytes)
                print(f'[Temperature] Manual raster saved -> {dest}')
            elif suffix == '.zip':
                raw_path = f'/tmp/{country}_temp_manual.zip'
                with open(raw_path, 'wb') as f:
                    f.write(content_bytes)
                folder_name = os.path.splitext(fname)[0]
                dest_folder = f'{out}/{folder_name}'
                with zipfile.ZipFile(raw_path, 'r') as z:
                    z.extractall(dest_folder)
                os.remove(raw_path)
                print(f'[Temperature] Manual GSA folder extracted -> {dest_folder}')
            else:
                raise Exception(f'Unsupported file type "{suffix}". Use .tif/.tiff, or a .zip of the GSA folder.')
            _record_manual_success(country, 'Temperature (Solar Atlas)')
            temp_upload_status.value = (f'<span style="color:#188038;">Manual Temperature raster for '
                                         f'<b>{country}</b> loaded from "{fname}".</span>')
        except Exception as e:
            print(f'Could not save uploaded Temperature file: {e}')
            temp_upload_status.value = f'<span style="color:#d93025;">Upload failed: {e}</span>'

temp_upload_btn.on_click(on_temp_upload_click)

def make_card(label, desc, tag='Optional'):
    tc  = '#1a73e8' if tag == 'Required' else '#5f6368'
    tag_html = (f'<span style="float:right;font-size:10px;color:white;background:{tc};'
                f'padding:2px 8px;border-radius:10px;font-weight:bold;text-transform:uppercase;">'
                f'{tag}</span>')
    dw  = widgets.HTML(
        value=tag_html + f'<div style="margin-top:6px;font-size:13px;padding-right:80px;">{desc}</div>',
        layout=Layout(margin='4px 0 8px 0')
    )
    cb  = widgets.Checkbox(description=label, value=False, indent=False)
    box = VBox([cb, dw], layout=Layout(width='100%', padding='14px'))
    box.add_class('sdk-card')
    return box, cb

all_cb = widgets.Checkbox(description='Select / deselect all datasets', value=False, indent=False)

pop_box,    pop_cb    = make_card('Population (WorldPop 1 km)',
    'Gridded population counts for 2020 at 1 km resolution. The base grid layer for all OnSTOVE analysis.',
    tag='Required')
urb_box,    urb_cb    = make_card('Urban-Rural status (GHS-SMOD 2020)',
    'Settlement classification used to calibrate stove shares and electrification rates for urban vs rural areas.',
    tag='Required')
mv_box,     mv_cb     = make_card('Medium-voltage lines (Gridfinder ~724 MB)',
    'Predicted global MV grid network (Zenodo CC-BY 4.0). Used with NTL and population to calibrate electrified settlements.',
    tag='Required')
hv_box,     hv_cb     = make_card('HV lines (OpenStreetMap via Overpass, power=line)',
    'Existing high-voltage electricity TRANSMISSION lines only - no roads, no lower-voltage/minor '
    'lines, no other infrastructure. Used as a distance-to-grid proxy. Coverage depends on how well '
    'the country\'s grid is mapped in OSM; see Instructions.txt for a manual alternative.')
ntl_box,    ntl_cb    = make_card('Nighttime lights (WorldPop Global2 VIIRS NVF 2020)',
    'Annual average radiance composite. Combined with population density and grid infrastructure in the MCA electrification calibration.',
    tag='Required')
walk_box,   walk_cb   = make_card('Walking friction (MalariaAtlas 202001)',
    'Time to walk 1 metre per sq-km. Drives least-cost path estimates for firewood and manure collection times.')
moto_box,   moto_cb   = make_card('Motorized friction (MalariaAtlas 202001)',
    'Time to travel 1 metre by vehicle per sq-km. Used to estimate LPG delivery cost and transport emissions.')
travel_box, travel_cb = make_card('Travel time to nearest large town (MalariaAtlas method)',
    'Least-cost travel time (minutes) to the nearest large town, computed the same way as the MalariaAtlas Accessibility to Cities surface. Used as a proxy for travel time to LPG supply points.')
forest_box, forest_cb = make_card('Forest cover (Hansen/GLAD GFC 2020, optical)',
    'Tree canopy cover tiles from the Global Forest Change dataset. Used to locate biomass supply and estimate collection time.')
forest_ee_box, forest_ee_cb = make_card('Forest cover (PALSAR FNF 2021, radar - optional)',
    'JAXA ALOS PALSAR-2 forest/non-forest classification via Google Earth Engine. Radar-based, so it '
    'sees through cloud/haze where optical layers like the one above can have gaps (e.g. Congo Basin). '
    'Requires a one-time Earth Engine login the first time it runs this session. Research/non-commercial '
    'use only per JAXA\'s data terms.',
    tag='Optional - needs login')
live_box,   live_cb   = make_card('Livestock density (FAO GLW4 2020 - cattle goats sheep)',
    'Gridded animal density from FAO. Used to estimate manure availability per grid cell for household biogas production.')
temp_box,   temp_cb   = make_card('Temperature - full GIS package (Global Solar Atlas v2)',
    'Downloads the entire Global Solar Atlas country GEOTIFF folder (GHI, DNI, DIF, GTI, PVOUT, TEMP, ELE, OPTA + metadata) rather than just TEMP.tif. Air temperature drives LPG/biogas suitability and cooking demand calibration.')
rwi_box,    rwi_cb    = make_card('Relative Wealth Index (Meta / HDX)',
    'Sub-national wealth index at ~2.4 km. Used to distribute minimum wage spatially to monetise time savings from clean cooking.')
specs_box,  specs_cb  = make_card('Blank specification CSVs',
    'Generates blank socio-economic and techno-economic CSV templates with all OnSTOVE parameters pre-labelled.',
    tag='Required')
tmpl_box,   tmpl_cb   = make_card('Specification templates (Technoeconomic + Prep file, XLSX)',
    'Blank, generic Technoeconomic and Prep file workbooks - not country data. Each includes a Read Me and a Parameter Legend tab, and applies to any country.')

ALL_CBS = [pop_cb, urb_cb, mv_cb, hv_cb, ntl_cb, walk_cb,
           moto_cb, travel_cb, forest_cb, forest_ee_cb, live_cb, temp_cb, rwi_cb, specs_cb, tmpl_cb]

def on_all(change):
    for cb in ALL_CBS: cb.value = all_cb.value
all_cb.observe(on_all, names='value')

summary = widgets.Label(value='Total datasets to fetch: 0')
def upd(_):
    summary.value = f'Total datasets to fetch: {sum(c.value for c in ALL_CBS)}'
for cb in ALL_CBS: cb.observe(upd, names='value')

btn = Button(description=' Get Data', icon='cloud-download',
             button_style='primary', layout=Layout(width='180px', height='42px'))
log = widgets.Output()

TASK_MAP = [
    (pop_cb,    'Population',         lambda c: get_population(c)),
    (urb_cb,    'Urban-Rural',         lambda c: get_urban_rural(c)),
    (mv_cb,     'MV lines',            lambda c: get_mv_lines(c)),
    (hv_cb,     'HV lines',            lambda c: get_hv_lines(c)),
    (ntl_cb,    'Nighttime Lights',    lambda c: get_nighttime_lights(c)),
    (walk_cb,   'Walking Friction',    lambda c: get_walking_friction(c)),
    (moto_cb,   'Motorized Friction',  lambda c: get_motorized_friction(c)),
    (travel_cb, 'Travel Time to Towns', lambda c: get_travel_time_to_towns(c)),
    (forest_cb, 'Forest Cover',        lambda c: get_forest_cover(c)),
    (forest_ee_cb, 'Forest Cover (PALSAR FNF)', lambda c: get_forest_cover_palsar_fnf(c, ee_project_w.value.strip() or None)),
    (live_cb,   'Livestock',           lambda c: get_livestock(c)),
    (temp_cb,   'Temperature (Solar Atlas)', lambda c: get_temperature_solaratlas(c)),
    (rwi_cb,    'Wealth Index',        lambda c: get_wealth_index(c)),
    (specs_cb,  'Blank CSVs',          lambda c: get_blank_csvs(c)),
    (tmpl_cb,   'Spec Templates (XLSX)', lambda c: get_specification_templates(c)),
]

# Tracks what happened on the most recent run per country, so Step 4 can write
# an Instructions file summarising it (requested datasets, which failed and why).
RUN_STATUS = {}

# Manual-fallback pointers for datasets that have no reliable public API, used
# when writing the Instructions file - both for datasets that failed this run
# and for the ones that are always manual (Electricity Transformers).
MANUAL_FALLBACK = {
    'Boundary': ('GADM boundary (required by most other layers).',
                 'https://gadm.org/download_country.html -> select country -> GeoPackage/Shapefile/GeoJSON '
                 '(level-0/whole-country only), OR use the "Upload boundary manually" button in Step 3.'),
    'Temperature (Solar Atlas)': ('Global Solar Atlas full GIS package.',
                 'https://globalsolaratlas.info/download -> select country -> download the '
                 'GIS_LTAym_YearlyMonthlyTotals GEOTIFF zip -> place the WHOLE unzipped folder in '
                 'Data/<ISO3>/Temperature/'),
    'Wealth Index': ('Relative Wealth Index or WorldPop poverty layer.',
                 'https://data.humdata.org/dataset/relative-wealth-index (per-country CSV or 93-country zip) '
                 'or https://hub.worldpop.org/project/categories?id=9 (WorldPop poverty index)'),
    'HV lines': ('OSM power=line (high-voltage transmission lines ONLY - no roads, no lower-voltage '
                 'lines) via Overpass is used automatically, but coverage is only as complete as OSM '
                 'mapping in-country. Where available, a curated national transmission network shapefile '
                 'is a better source.',
                 'https://energydata.info -> search "<country name> transmission lines / grid network", '
                 'OR https://www.geofabrik.de/data/energy-networks.html for a paid custom OSM power export.'),
}
ALWAYS_MANUAL = {
    'Electricity Transformers': ('Country-specific infrastructure data - no public API.',
                 'https://energydata.info or your own national utility/regulator data.'),
}

def on_click(_):
    country = country_w.value
    if country == 'Select country':
        with log: print('Please select a country first.')
        return
    btn.disabled = True; btn.description = ' Fetching...'
    btn.icon = 'spinner'; btn.button_style = 'warning'
    tasks = [(lbl, fn) for (cb, lbl, fn) in TASK_MAP if cb.value]
    log.clear_output()
    with log:
        print(f'--- Starting download for {country} ---')
        try:
            get_boundaries(country)
        except Exception as e:
            print(f'\nERROR: Could not fetch the country boundary, so the run has been stopped: {e}')
            print('The boundary is a required first step (most other layers are clipped to it).')
            print('This is usually a transient GADM server issue - wait a few minutes and click Get Data again,')
            print('or Runtime > Restart runtime if it keeps happening.')
            RUN_STATUS[country] = {
                'requested': [lbl for lbl, fn in tasks],
                'succeeded': [],
                'failed': [('Boundary', str(e))],
            }
            import time
            btn.icon = 'exclamation-triangle'; btn.description = ' Failed'; btn.button_style = 'danger'
            time.sleep(3)
            btn.description = ' Get Data'; btn.icon = 'cloud-download'
            btn.button_style = 'primary'; btn.disabled = False
            return
        failed = []
        succeeded = []
        for lbl, fn in tqdm(tasks, desc='Downloading datasets'):
            try:
                fn(country)
                succeeded.append(lbl)
            except Exception as e:
                print(f'\nWARNING [{lbl}]: {e}')
                failed.append((lbl, str(e)))
            finally:
                gc.collect()
        RUN_STATUS[country] = {
            'requested': [lbl for lbl, fn in tasks],
            'succeeded': succeeded,
            'failed': failed,
        }
        if failed:
            print(f'\nDatasets with errors: {", ".join(lbl for lbl, _ in failed)}')
            print('You can retry or upload these manually.')
        print(f'\nDownloads complete for {country}!')
        print(f'Files saved in: Data/{country}/')
        print('Proceed to Step 4 to package your zip (an Instructions file summarising this run')
        print('and any missing datasets will be included automatically).')
    import time
    btn.icon = 'check'; btn.description = ' Data Ready'; btn.button_style = 'success'
    time.sleep(3)
    btn.description = ' Get Data'; btn.icon = 'cloud-download'
    btn.button_style = 'primary'; btn.disabled = False

btn.on_click(on_click)

manual_note = widgets.HTML(value=(
    '<div class="sdk-warn">'
    '<b>Layers requiring manual download (no public API)</b><br><br>'
    'Electricity transformers &rarr; country-specific - use <a href="https://energydata.info" target="_blank">EnergyData.info</a> or your own national data<br><br>'
    '<b>Note:</b> Temperature is now fetched automatically as the full Global Solar Atlas GEOTIFF folder '
    '(GHI, DNI, DIF, GTI, PVOUT, TEMP, ELE, OPTA + metadata) - not just <code>TEMP.tif</code>. '
    'If the automatic download fails for your country, fall back to '
    '<a href="https://globalsolaratlas.info/download" target="_blank">Global Solar Atlas</a> manually.<br><br>'
    '<b>Note:</b> PALSAR FNF forest cover is the only layer needing a login - a one-time Earth Engine '
    'authentication prompt will appear if that box is ticked.'
    '</div>'
))

req_acc = widgets.Accordion(children=[VBox([pop_box, urb_box, ntl_box, mv_box, specs_box, tmpl_box])])
req_acc.set_title(0, 'Required Layers'); req_acc.add_class('borderless')

ee_project_w = widgets.Text(
    value='', placeholder='e.g. my-earth-engine-project',
    description='EE project:', layout=Layout(width='320px'),
    style={'description_width': '80px'}
)
ee_project_note = widgets.HTML(value=(
    '<span style="color:#5f6368;font-size:12px;">'
    'Only needed for the PALSAR FNF (radar) forest cover box above. Must be a Google Cloud project '
    'with the Earth Engine API enabled - register one free at '
    '<a href="https://code.earthengine.google.com/" target="_blank">code.earthengine.google.com</a> '
    'if you don\'t have one. Leave blank to use Earth Engine\'s default project detection.'
    '</span>'
))

geo_acc = widgets.Accordion(children=[VBox([forest_box, forest_ee_box, HBox([ee_project_w]), ee_project_note,
                                             live_box, temp_box, walk_box, moto_box, travel_box])])
geo_acc.set_title(0, 'Physical and Environmental Layers'); geo_acc.add_class('borderless')

infra_acc = widgets.Accordion(children=[VBox([hv_box, rwi_box])])
infra_acc.set_title(0, 'Infrastructure and Socioeconomic Layers'); infra_acc.add_class('borderless')

app = VBox([
    widgets.HTML('<h2 style="color:#1a73e8;font-family:Segoe UI,sans-serif;">OnSTOVE Starter Data Kit</h2>'),
    widgets.HTML('<br>'),
    HBox([country_w]),
    widgets.HTML('<br>'),
    VBox([all_cb], layout=Layout(padding='10px')).add_class('sdk-all-box'),
    req_acc, geo_acc, infra_acc,
    manual_note, summary,
    widgets.HTML('<br><b>Boundary fetch failing?</b>'),
    HBox([upload_btn]), upload_status,
    widgets.HTML('<br><b>Temperature raster fetch failing?</b>'),
    HBox([temp_upload_btn]), temp_upload_status,
    widgets.HTML('<b>Console output</b>'),
    log,
    HBox([btn], layout=Layout(justify_content='center', padding='20px'))
], layout=Layout(max_width='1050px', margin='0 auto'))

display(app)

In [4]:
# @title Forest Cover Fetch {"display-mode": "form"}
# @markdown For some countries, forest cover is such a large dataset that this step
# @markdown can crash the Colab runtime. We recommend running this cell **after** all
# @markdown other datasets have been collected in Step 3, so that a crash here doesn't
# @markdown take the rest of your downloads with it. If it crashes, just restart the
# @markdown runtime and re-run this cell on its own.

country = country_w.value # Get the selected country from the dropdown
with log:
    print(f'--- Re-running Forest Cover download for {country} ---')
    try:
        get_forest_cover(country)
        # Update RUN_STATUS: remove from failed, add to succeeded
        if country in RUN_STATUS:
            RUN_STATUS[country]['failed'] = [(lbl, err) for lbl, err in RUN_STATUS[country]['failed'] if lbl != 'Forest Cover']
            if 'Forest Cover' not in RUN_STATUS[country]['succeeded']:
                RUN_STATUS[country]['succeeded'].append('Forest Cover')
        print(f'\nForest Cover download successful for {country}!')
    except Exception as e:
        print(f'\nWARNING [Forest Cover]: {e}')
        # Update RUN_STATUS: add to failed if not already there
        if country in RUN_STATUS:
            if 'Forest Cover' not in [lbl for lbl, _ in RUN_STATUS[country]['failed']]:
                RUN_STATUS[country]['failed'].append(('Forest Cover', str(e)))
        print(f'Forest Cover download failed for {country}. Please check the error message above.')
    finally:
        gc.collect()

In [5]:
# @title Step 4: Package and download {"display-mode": "form"}
# @markdown Zips all downloaded data and saves it as **<ISO3>_SDK.zip**.
# @markdown An **Instructions.txt** file is generated and included automatically,
# @markdown listing next steps and any datasets that were missing or failed to
# @markdown download for that country.
# @markdown Make sure Step 3 has completed first.

import os, zipfile, glob, datetime
from google.colab import files as cfiles

countries = [d.split('/')[-1] for d in glob.glob('Data/*') if os.path.isdir(d)]

def _build_instructions(country):
    status = RUN_STATUS.get(country) if 'RUN_STATUS' in globals() else None
    lines = []
    lines.append(f'OnSTOVE Starter Data Kit - Instructions for {country}')
    lines.append(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
    lines.append('=' * 60)
    lines.append('')
    lines.append('NEXT STEPS')
    lines.append('-' * 60)
    lines.append('1. Fill in the blank socio-economic and techno-economic CSVs/templates')
    lines.append('   in Specifications/ with country-specific values.')
    lines.append('2. Manually add Electricity Transformers:')
    manual_desc, manual_url = ALWAYS_MANUAL['Electricity Transformers']
    lines.append(f'   {manual_desc}')
    lines.append(f'   {manual_url}')
    lines.append('3. See https://onstove-documentation.readthedocs.io to run the model.')
    lines.append('')

    if status is None:
        lines.append('NOTE: No record of this run was found (e.g. this country was downloaded in a')
        lines.append('previous session). Re-run Step 3 for this country to get a full dataset summary here.')
    else:
        succeeded = status.get('succeeded', [])
        failed = status.get('failed', [])
        lines.append('DATASETS DOWNLOADED SUCCESSFULLY')
        lines.append('-' * 60)
        if succeeded:
            for lbl in succeeded:
                lines.append(f'  [OK] {lbl}')
        else:
            lines.append('  (none recorded)')
        lines.append('')

        if failed:
            lines.append('DATASETS THAT FAILED OR ARE MISSING - ACTION NEEDED')
            lines.append('-' * 60)
            for lbl, err in failed:
                lines.append(f'  [MISSING] {lbl}')
                lines.append(f'    Error: {err}')
                if lbl in MANUAL_FALLBACK:
                    desc, url = MANUAL_FALLBACK[lbl]
                    lines.append(f'    Manual fallback: {desc}')
                    lines.append(f'    {url}')
                else:
                    lines.append('    Try re-running Step 3 for this country - this may be a transient')
                    lines.append('    server issue.')
                lines.append('')
        else:
            lines.append('DATASETS THAT FAILED OR ARE MISSING')
            lines.append('-' * 60)
            lines.append('  None - everything requested downloaded successfully.')
            lines.append('')

    lines.append('ALWAYS MANUAL (no public API - not attempted automatically)')
    lines.append('-' * 60)
    for lbl, (desc, url) in ALWAYS_MANUAL.items():
        lines.append(f'  {lbl}: {desc}')
        lines.append(f'  {url}')
    lines.append('')

    return '\n'.join(lines)

if not countries:
    print('No data found in Data/ - please complete Step 3 first.')
else:
    for country in countries:
        instr_path = f'Data/{country}/Instructions.txt'
        with open(instr_path, 'w') as f:
            f.write(_build_instructions(country))
        print(f'[Instructions] Written -> {instr_path}')

        zname = f'{country}_SDK.zip'
        print(f'Packaging {country} -> {zname} ...')
        with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as zf:
            for root, _, files in os.walk(f'Data/{country}'):
                for fname in files:
                    fpath = os.path.join(root, fname)
                    zf.write(fpath, os.path.relpath(fpath, '/content/'))
        mb = os.path.getsize(zname) / 1048576
        print(f'  {zname}  ({mb:.1f} MB)')
        cfiles.download(zname)
        print(f'  Download started for {zname}')
    print('\nAll done! Your OnSTOVE starter data kit is ready.')
    print('See Instructions.txt inside each zip for next steps and any datasets you need to add manually.')


[Instructions] Written -> Data/Select country/Instructions.txt
Packaging Select country -> Select country_SDK.zip ...
  Select country_SDK.zip  (0.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Download started for Select country_SDK.zip

All done! Your OnSTOVE starter data kit is ready.
See Instructions.txt inside each zip for next steps and any datasets you need to add manually.


In [6]:
# @title Optional: Delete session data {"display-mode": "form"}
# @markdown Removes downloaded data from the Colab session to free disk space.

import glob, os, shutil
import ipywidgets as widgets
from ipywidgets import HBox, Layout, Button
from IPython.display import display

def loaded():
    return [d.split('/')[-1] for d in glob.glob('Data/*') if os.path.isdir(d)] or ['(none)']

dd  = widgets.Dropdown(description='Country:', options=loaded(),
                       layout=Layout(width='240px'), style={'description_width':'80px'})
del_btn = Button(description='Delete data', icon='trash',
                 layout=Layout(width='160px', height='35px'), button_style='danger')

def on_del(_):
    c = dd.value
    if c == '(none)': print('Nothing to delete.'); return
    shutil.rmtree(f'Data/{c}', ignore_errors=True)
    print(f'Data for {c} deleted.')
    dd.options = loaded()

del_btn.on_click(on_del)
display(HBox([dd, del_btn]))

Data for Select country deleted.
